# AI-Harm Single-Pass Inference + Full PHTKG Benchmark

**Inference contract:** one curated report → one Qwen extraction call → one structured event.
The upstream `classification`/`classifications` value is preserved as `harm_category`.


### Auto-PHTKG edition

This notebook is the self-parametric version. Set `AUTO_TUNE=True` to search validation-safe hyperparameters automatically. The tuner does **not** inspect test results; after selection it freezes the best configuration, retrains the two final experiments, saves the winning config/trial table, and then runs the original diagnostics.


In [ ]:
# Run only in a fresh Colab environment when dependencies are missing.
# !pip install -q pandas numpy torch transformers accelerate sentencepiece openpyxl

## 1. Imports and configuration

Ikram Implemetation

In [ ]:
import gc
import copy
import hashlib
import json
import math
import random
import re
import resource
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

INPUT_PATH = Path("/content/new_combined_data (1).json")
GOLD_PATH = Path("/content/R001_R1162_human_annotated_clean_corrected.json")

# if not INPUT_PATH.exists():
#     INPUT_PATH = Path("/mnt/data/ground_truth_dataset(1).json")
# if not GOLD_PATH.exists():
#     GOLD_PATH = Path("/mnt/data/ground_truth_entities_only(1).json")

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/AI_Harm_Map_ICLR")
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Persistent output directory:", OUTPUT_DIR)
TAXONOMY_PATH = DRIVE_OUTPUT_DIR / "AI_Harm_Map_Taxonomy_Schema_vSHARED (1).xlsx"
TAXONOMY_SHEET = "Taxonomy & Schema"

SEED = 42
RUN_QWEN = False
RUN_NLI = False  # WIP single-pass: no NLI verifier/gate
MAX_RECORDS = None  # None = run inference on the full input corpus

# Qwen inference speed/quality controls.
# Keep generous input/output ceilings for extraction completeness; speed comes
# from batching + SDPA rather than truncating the answer.
QWEN_BATCH_SIZE = 4          # Start at 4 on a Colab T4; adaptive OOM fallback splits batches automatically.
QWEN_MAX_INPUT_TOKENS = 7000
QWEN_MAX_NEW_TOKENS = 700    # Ceiling only; generation stops earlier on EOS.
QWEN_USE_SDPA = True
RESUME_EXTRACTION = True

QWEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# -----------------------------------------------------------------------------
# PHTKG defaults + Auto-PHTKG controls
# -----------------------------------------------------------------------------
# These values remain the fallback configuration.  When AUTO_TUNE=True, the
# tuner below searches a leakage-safe validation objective, applies the best
# configuration, and then trains the final gold/predicted models.
EMBEDDING_DIM = 64
LAYERS = 2
EPOCHS = 120
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 3e-4
DROPOUT = 0.10
LABEL_SMOOTHING = 0.0
NEGATIVES = 6
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
PATIENCE = 25
EXTRAPOLATION_WEIGHT = 0.10
EXTRAPOLATION_WARMUP_EPOCHS = 30
RANKING_MARGIN = 0.25
RANKING_WEIGHT = 0.65
GRAD_CLIP = 1.5
HARD_NEGATIVE_POOL_MULTIPLIER = 1
LR_SCHEDULER_PATIENCE = 6

# Self-parametric search.  The TEST split is never evaluated inside tuning.
AUTO_TUNE = True
AUTO_TUNE_TRIALS = 30
AUTO_TUNE_EPOCHS = 100
AUTO_TUNE_PATIENCE = 12
AUTO_TUNE_SEED = SEED + 7000
AUTO_TUNE_STD_PENALTY = 0.15  # rewards stable paired-AUC across corruption draws

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Input:", INPUT_PATH)
print("Gold:", GOLD_PATH)

## Auto-PHTKG (leakage-safe)

This version keeps the PHTKG roles, provenance mechanism, temporal recurrence head, and rule-based corruption family, while adding **self-parametric model selection** around the existing training procedure.

Correctness constraints remain in force:

1. **Incident-only entity updates** — entities absent from a bucket preserve their state instead of drifting via `GRU(0, h)`.
2. **Leakage-safe negative guard** — training-time corruptions are checked against **training positives only**.
3. **Post-update state synchronization** — chronological entity state is recomputed with the updated checkpoint weights before validation.
4. **Untouched test during tuning** — Auto-PHTKG selects hyperparameters only from validation paired-AUC; test scores are produced only by the final fit.

Auto-PHTKG searches model capacity, optimization, regularization, ranking/recurrence weights, negative-sampling intensity, hard-negative mining, and corruption-strategy mixture.  Trial 1 is always the original hand-set configuration, so automatic search cannot silently discard the current baseline.


## 2. Single-pass extraction prompt

One Qwen call extracts the event fields. The supplied classification is contextual metadata only
and is copied by Python after generation; Qwen does not predict or verify it.


In [ ]:
EXTRACTION_PROMPT = r"""
You are the event-extraction model inside a multilingual AI-harm knowledge-graph pipeline.

TASK
----
The corpus has already been curated as AI-harm reports. Do NOT decide whether the report
"qualifies" as an AI-harm event. Extract ONE central event record from the report.

This is a SINGLE-PASS inference task. Search the ENTIRE report before declaring any field
unavailable. Preserve source terminology and attribution. Do not invent facts.

IMPORTANT
---------
- The report may already contain a supplied classification.
- You may use that supplied classification only as contextual guidance for identifying the
  central event.
- DO NOT predict, rewrite, verify, replace, or output the harm category.
- Python will copy the input classification mechanically after this model call.

REQUIRED FIELDS
---------------
event_type
ai_system
organization
affected_group
action
consequence
location
event_date
confidence

GLOBAL RULES
------------
1. Return exactly ONE event and valid JSON only.
2. Never return JSON null.
3. If a factual field genuinely cannot be established after reading the full report, use exactly:
   "Not specified in report"
4. Do not omit any required field.
5. Preserve whether claims are alleged, reported, found by an audit, denied, disputed, etc.
6. Prefer exact source wording for named entities.
7. A commercial/product name is NOT required for ai_system; a supported generic descriptor
   such as "AI chatbot", "facial recognition technology", "automated hiring algorithm",
   "AI system", or "recommender system" is valid.
8. Do not confuse the article publisher with the organization responsible for the event.

AI SYSTEM
---------
Extract the AI system, model, algorithm, automated tool, or AI-enabled process actually involved.
Search the whole report. Generic source-supported descriptions are valid.

ORGANIZATION
------------
Extract the organization, authority, company, institution, platform, government body, or actor
that deployed, operated, commissioned, purchased, used, or was responsible for the AI-enabled
process. If several actors are directly involved, describe them concisely using source-supported
names.

AFFECTED GROUP
--------------
Extract the most specific person/group/community exposed to or affected by the action.
Do not substitute the organization. Use a broader description only if the report itself supports it.

ACTION
------
State what the AI-enabled system/process actually did. Keep this distinct from the harm.
Examples: ranked applicants, flagged claims, identified people, generated content, recommended
content, monitored participants, classified individuals, predicted risk.

CONSEQUENCE
-----------
State the adverse outcome, risk, exposure, restriction, denial, inequality, privacy intrusion,
safety problem, exploitation, discrimination, surveillance, misinformation effect, or other
negative consequence associated with the action.
Do NOT use neutral background activity as a consequence.
Do NOT merely repeat the action.

EVENT TYPE
----------
Write a short 2–8 word descriptive title for the concrete event/harm.
Do not use generic labels such as "ai_harm_event".

DATE
----
Use this priority:
1. explicit event date;
2. explicit month/year;
3. explicit date range;
4. publication date as fallback.
Return YYYY-MM-DD, YYYY-MM, or YYYY. Do not invent missing components.

LOCATION
--------
Use the most specific source-supported location/jurisdiction/platform context.
Do not invent a location.

CONFIDENCE
----------
Return a number from 0 to 1 reflecting how directly the extracted event fields are supported.

FINAL CHECK
-----------
Before returning JSON, verify internally that:
- you searched the whole report for ai_system, organization, affected_group, action,
  consequence, location, and date;
- action and consequence are semantically different roles;
- no unsupported fact was introduced;
- every required field is present;
- harm_category is NOT included in your generated event.

OUTPUT FORMAT
-------------
Return exactly:

{
  "harm_event": {
    "event_type": "...",
    "ai_system": "...",
    "organization": "...",
    "affected_group": "...",
    "action": "...",
    "consequence": "...",
    "location": "...",
    "event_date": "...",
    "confidence": 0.0
  }
}

Return JSON only.
"""


## 3. Load and validate the aligned input and gold files

In [ ]:
def clean(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()

def normalize(value):
    return " ".join(re.sub(r"[^\w\s]", " ", clean(value).casefold()).split())

def year_from(*values):
    for value in values:
        match = re.search(r"\b(?:19|20)\d{2}\b", clean(value))
        if match:
            return int(match.group(0))
    return 0

def first_json(text):
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object returned.")
    depth = 0
    in_string = False
    escaped = False
    for index, char in enumerate(text[start:], start):
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return json.loads(text[start:index + 1])
    raise ValueError("Incomplete JSON object.")

def _missing(value):
    return value is None or value == "" or value == []

def supplied_classification(report):
    """Return the upstream classification exactly as supplied; never predict it."""
    raw = report.get("classifications")
    if _missing(raw):
        raw = report.get("classification")
    return copy.deepcopy(raw) if not _missing(raw) else []

def graph_text(value):
    """Deterministic string representation for graph vocabulary only."""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return clean(value)

reports = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
gold_events = json.loads(GOLD_PATH.read_text(encoding="utf-8"))

if MAX_RECORDS is not None:
    reports = reports[:MAX_RECORDS]

report_ids = [row["report_id"] for row in reports]
gold_ids = [row["report_id"] for row in gold_events]

assert len(report_ids) == len(set(report_ids)), "Duplicate input report_id values."
assert len(gold_ids) == len(set(gold_ids)), "Duplicate gold report_id values."

reports_by_id = {row["report_id"]: row for row in reports}
gold_by_id = {row["report_id"]: row for row in gold_events}

# Gold is evaluation-only. It must never limit inference coverage.
overlap_ids = set(report_ids) & set(gold_ids)
gold_events_eval = [row for row in gold_events if row["report_id"] in overlap_ids]

def parse_harm_category(value):
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    if isinstance(value, str) and value.strip():
        parts = [p.strip() for p in value.split(";") if p.strip()]
        parsed = []
        for part in parts:
            try:
                parsed.append(json.loads(part))
            except json.JSONDecodeError:
                # Preserve a non-JSON category string rather than silently losing it.
                parsed.append(part)
        return parsed
    return []

for row in gold_events:
    row["harm_category"] = parse_harm_category(row.get("harm_category"))

print("Inference input reports:", len(reports))
print("Gold annotations available:", len(gold_events))
print("Gold/input overlap for evaluation:", len(gold_events_eval))
print("Input reports without gold (still inferred):", len(set(report_ids) - set(gold_ids)))
display(pd.DataFrame(reports).head(3))
display(pd.DataFrame(gold_events_eval).head(3))


## 4.  Qwen inference and checkpointing

Each report is still extracted in **one inference pass**. Reports are batched together on the GPU
for throughput, but each report produces its own independent JSON completion.

Speedups used here:
- batched generation (`QWEN_BATCH_SIZE=4` by default);
- PyTorch SDPA attention when available;
- KV cache during generation;
- adaptive CUDA-OOM fallback that splits a batch instead of failing the run;
- resumable JSONL checkpoints.



In [ ]:
class QwenExtractor:
    """Fast single-pass Qwen extractor with batched deterministic generation."""

    def __init__(self, model_name):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        print("Loading Qwen model...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if torch.cuda.is_available():
            load_kwargs = {
                "device_map": "auto",
                "dtype": torch.float16,
            }
            if QWEN_USE_SDPA:
                load_kwargs["attn_implementation"] = "sdpa"

            try:
                self.model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    **load_kwargs,
                )
                if QWEN_USE_SDPA:
                    print("Qwen attention: SDPA")
            except Exception as error:
                if QWEN_USE_SDPA:
                    print(
                        "SDPA load failed; retrying with the Transformers default attention. "
                        f"Reason: {type(error).__name__}: {error}"
                    )
                    load_kwargs.pop("attn_implementation", None)
                    self.model = AutoModelForCausalLM.from_pretrained(
                        model_name,
                        **load_kwargs,
                    )
                else:
                    raise
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.float32,
            )

        self.model.eval()
        print("Qwen model loaded.")

    def _build_prompt(self, report):
        # IMPORTANT: supplied classification is deliberately NOT included here.
        # It is upstream data, not something Qwen needs to infer.
        user_text = f"""
REPORT ID:
{clean(report.get("report_id"))}

SOURCE:
{clean(report.get("source_url"))}

PUBLICATION DATE:
{clean(report.get("publication_date"))}

SOURCE LANGUAGE:
{clean(report.get("source_language"))}

COUNTRY / DATASET LOCATION HINT:
{clean(report.get("country"))}

ORIGINAL-LANGUAGE REPORT:
{clean(report.get("original_text"))}

CANONICAL ENGLISH TRANSLATION:
{clean(report.get("translated_text"))}
""".strip()

        messages = [
            {"role": "system", "content": EXTRACTION_PROMPT},
            {"role": "user", "content": user_text},
        ]

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def _normalize_completion(self, report, generated_text, generated_token_count, batch_size):
        parsed = first_json(generated_text)
        event = parsed.get("harm_event")

        if not isinstance(event, dict):
            raise ValueError("Qwen did not return a harm_event object.")

        fields = [
            "event_type",
            "ai_system",
            "organization",
            "affected_group",
            "action",
            "consequence",
            "location",
            "event_date",
            "confidence",
        ]

        normalized = {}
        for field in fields:
            value = event.get(field, "Not specified in report")
            if value is None or value == "":
                value = "Not specified in report" if field != "confidence" else 0.0
            normalized[field] = value

        original_text = clean(report.get("original_text"))
        translated_text = clean(report.get("translated_text"))
        source_language = clean(report.get("source_language")).lower()

        normalized["original_evidence_span"] = (
            original_text if original_text else "Not specified in report"
        )
        normalized["translated_evidence_span"] = (
            original_text if source_language == "en" and original_text
            else translated_text if source_language != "en" and translated_text
            else "Not specified in report"
        )
        normalized["original_text"] = original_text
        normalized["translated_text"] = (
            original_text if source_language == "en" else translated_text
        )

        normalized["harm_category"] = copy.deepcopy(supplied_classification(report))
        normalized["report_id"] = report["report_id"]
        normalized["source"] = clean(report.get("source_url")) or "Not specified in report"
        normalized["evidence_audit"] = {
            "method": "full_source_text_copy",
            "source_field": "original_text",
        }
        normalized["inference_audit"] = {
            "mode": "single_pass_batched",
            "generated_tokens": int(generated_token_count),
            "batch_size": int(batch_size),
            "max_input_tokens": int(QWEN_MAX_INPUT_TOKENS),
            "max_new_tokens": int(QWEN_MAX_NEW_TOKENS),
        }

        return normalized

    def extract_batch(self, reports_batch):
        """Generate one independent completion per report, batched for GPU throughput."""
        if not reports_batch:
            return []

        prompts = [self._build_prompt(report) for report in reports_batch]

        encoded = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=QWEN_MAX_INPUT_TOKENS,
        )

        device = next(self.model.parameters()).device
        encoded = {key: value.to(device) for key, value in encoded.items()}
        input_width = encoded["input_ids"].shape[1]

        with torch.inference_mode():
            generated = self.model.generate(
                **encoded,
                max_new_tokens=QWEN_MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id,
            )

        results = []
        for report, sequence in zip(reports_batch, generated):
            new_ids = sequence[input_width:]

            # pad_token == eos_token for Qwen in this notebook. Count only tokens
            # before the first EOS so the audit reflects actual answer length.
            ids_list = new_ids.detach().cpu().tolist()
            eos_id = self.tokenizer.eos_token_id
            try:
                eos_position = ids_list.index(eos_id)
                answer_ids = ids_list[:eos_position]
            except ValueError:
                answer_ids = ids_list

            generated_text = self.tokenizer.decode(
                answer_ids,
                skip_special_tokens=True,
            )

            try:
                event = self._normalize_completion(
                    report,
                    generated_text,
                    len(answer_ids),
                    len(reports_batch),
                )
                results.append({"event": event, "error": None})
            except Exception as error:
                results.append({
                    "event": None,
                    "error": {
                        "report_id": report["report_id"],
                        "error": f"{type(error).__name__}: {error}",
                        "generated_text": generated_text[:2000],
                    },
                })

        # Release prompt tensors promptly between batches.
        del encoded, generated
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return results


def _is_cuda_oom(error):
    text = str(error).lower()
    return isinstance(error, torch.cuda.OutOfMemoryError) or (
        "cuda" in text and "out of memory" in text
    )


def extract_batch_adaptive(extractor, reports_batch):
    """Run a batch; on CUDA OOM, recursively split it without changing extraction logic."""
    try:
        return extractor.extract_batch(reports_batch)
    except RuntimeError as error:
        if not _is_cuda_oom(error):
            raise

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        if len(reports_batch) <= 1:
            raise

        midpoint = len(reports_batch) // 2
        left = reports_batch[:midpoint]
        right = reports_batch[midpoint:]
        print(
            f"CUDA OOM at batch size {len(reports_batch)}; "
            f"retrying as {len(left)} + {len(right)}.",
            flush=True,
        )
        return (
            extract_batch_adaptive(extractor, left)
            + extract_batch_adaptive(extractor, right)
        )


def use_gold_as_smoke_predictions():
    """Pipeline smoke test only; never report this as extraction performance."""
    return [dict(row) for row in gold_events_eval]


predictions_path = OUTPUT_DIR / "predicted_events_single_pass_fast.jsonl"

if RUN_QWEN:
    extractor = QwenExtractor(QWEN_MODEL)
    errors = []

    # Resume safely after a Colab disconnect without recomputing completed reports.
    if RESUME_EXTRACTION and predictions_path.exists():
        try:
            predictions = (
                pd.read_json(predictions_path, lines=True)
                .to_dict("records")
            )
            print("Resuming saved predictions:", len(predictions))
        except Exception as error:
            print(f"Could not read existing checkpoint; starting fresh: {error}")
            predictions = []
    else:
        predictions = []

    done_ids = {clean(row.get("report_id")) for row in predictions}
    remaining_reports = [
        report for report in reports
        if clean(report.get("report_id")) not in done_ids
    ]

    print("Remaining reports:", len(remaining_reports))
    print("Configured batch size:", QWEN_BATCH_SIZE)
    print("Max input tokens:", QWEN_MAX_INPUT_TOKENS)
    print("Max new tokens:", QWEN_MAX_NEW_TOKENS)

    for start in range(0, len(remaining_reports), QWEN_BATCH_SIZE):
        batch = remaining_reports[start:start + QWEN_BATCH_SIZE]
        batch_ids = [report["report_id"] for report in batch]
        print(
            f"[{start + 1}-{min(start + len(batch), len(remaining_reports))}/"
            f"{len(remaining_reports)}] {batch_ids}",
            flush=True,
        )

        try:
            batch_results = extract_batch_adaptive(extractor, batch)
        except Exception as error:
            # A generation-level failure should not erase prior progress.
            for report in batch:
                errors.append({
                    "report_id": report["report_id"],
                    "error": f"{type(error).__name__}: {error}",
                })
            print(f"Batch failed: {type(error).__name__}: {error}", flush=True)
            continue

        for item in batch_results:
            if item["event"] is not None:
                predictions.append(item["event"])
            else:
                errors.append(item["error"])

        # Keep output in input order for deterministic downstream behavior.
        order = {report["report_id"]: i for i, report in enumerate(reports)}
        predictions.sort(key=lambda row: order.get(row.get("report_id"), 10**12))

        # Checkpoint after every successful batch.
        pd.DataFrame(predictions).to_json(
            predictions_path,
            orient="records",
            lines=True,
            force_ascii=False,
        )

        if errors:
            pd.DataFrame(errors).to_csv(
                OUTPUT_DIR / "extraction_errors_single_pass_fast.csv",
                index=False,
            )

    # Output-token diagnostics: use these to decide later whether 700 can be reduced safely.
    output_token_counts = [
        row.get("inference_audit", {}).get("generated_tokens")
        for row in predictions
        if isinstance(row.get("inference_audit"), dict)
        and row.get("inference_audit", {}).get("generated_tokens") is not None
    ]
    if output_token_counts:
        print("\nQwen output-token statistics")
        print("Mean:", round(float(np.mean(output_token_counts)), 1))
        print("Median:", round(float(np.median(output_token_counts)), 1))
        print("P90:", round(float(np.percentile(output_token_counts, 90)), 1))
        print("P95:", round(float(np.percentile(output_token_counts, 95)), 1))
        print("P99:", round(float(np.percentile(output_token_counts, 99)), 1))
        print("Max:", int(max(output_token_counts)))

elif predictions_path.exists():
    predictions = (
        pd.read_json(predictions_path, lines=True)
        .to_dict("records")
    )
    print("Loaded saved predictions:", len(predictions))

else:
    predictions = use_gold_as_smoke_predictions()
    print("RUN_QWEN=False and no checkpoint exists.")
    print("Gold overlap is being used only to test downstream notebook mechanics.")

print(f"Extraction records available: {len(predictions)}")
print("Expected input records:", len(reports))
print("Inference contract: one extraction pass per report; reports may be GPU-batched for throughput.")


In [ ]:
if predictions:
    preview_index = min(11, len(predictions) - 1)
    print(json.dumps(predictions[preview_index], indent=2, ensure_ascii=False))
else:
    print("No predictions available.")


In [ ]:

# ============================================================
# 7. LOAD TAXONOMY + BUILD STRICT CLASS-FIT JOBS
# ============================================================

def resolve_if_missing(path, patterns):
    path = Path(path)
    if path.exists():
        return path
    candidates = []
    for pattern in patterns:
        candidates.extend(
            Path("/content/drive/MyDrive").rglob(pattern)
        )
    candidates = sorted(
        {p for p in candidates if p.is_file()},
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {path.name}. Edit TAXONOMY_PATH."
        )
    print("Auto-resolved taxonomy:", candidates[0])
    return candidates[0]

TAXONOMY_PATH = resolve_if_missing(
    TAXONOMY_PATH,
    ["AI_Harm_Map_Taxonomy_Schema_vSHARED*.xlsx"],
)

taxonomy_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name=TAXONOMY_SHEET,
).fillna("")
taxonomy_rows = taxonomy_df.to_dict("records")

def canonical_taxonomy_id(value):
    if value is None:
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return ""
        return f"{float(value):.10f}".rstrip("0").rstrip(".")
    return clean(value)

def classification_candidates(value):
    if value in (None, "", []):
        return []
    if isinstance(value, dict):
        return [copy.deepcopy(value)]
    if isinstance(value, list):
        return [copy.deepcopy(x) for x in value if x not in (None, "", [])]
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            return classification_candidates(json.loads(text))
        except Exception:
            pass

        parts = [p.strip() for p in text.split(";") if p.strip()]
        if len(parts) > 1:
            parsed = []
            all_json = True
            for part in parts:
                try:
                    parsed.extend(classification_candidates(json.loads(part)))
                except Exception:
                    all_json = False
                    break
            if all_json:
                return parsed

        return [text]

    return [copy.deepcopy(value)]

def candidate_id(candidate):
    if isinstance(candidate, dict):
        lower = {
            str(k).strip().casefold(): v
            for k, v in candidate.items()
        }
        for key in [
            "subcategory_id",
            "subcategory id",
            "#",
            "id",
            "taxonomy_id",
            "taxonomy id",
        ]:
            if key in lower and clean(lower[key]):
                return canonical_taxonomy_id(lower[key])

    if isinstance(candidate, (str, int, float)):
        text = canonical_taxonomy_id(candidate)
        if re.fullmatch(r"\d+(?:\.\d+)?", text):
            return text

    return ""

def candidate_name(candidate):
    if isinstance(candidate, dict):
        lower = {
            str(k).strip().casefold(): v
            for k, v in candidate.items()
        }
        for key in [
            "subcategory",
            "category",
            "harm_category",
            "harm category",
        ]:
            if key in lower and clean(lower[key]):
                return clean(lower[key])

    if isinstance(candidate, str):
        return candidate.strip()

    return ""

taxonomy_by_id = {}
taxonomy_by_name = {}

for row in taxonomy_rows:
    tid = canonical_taxonomy_id(row.get("#"))
    name = normalize(row.get("Subcategory"))
    if tid:
        taxonomy_by_id[tid] = row
    if name:
        taxonomy_by_name[name] = row

def match_taxonomy_row(candidate):
    tid = candidate_id(candidate)
    if tid and tid in taxonomy_by_id:
        return taxonomy_by_id[tid]

    name = normalize(candidate_name(candidate))
    if name and name in taxonomy_by_name:
        return taxonomy_by_name[name]

    if isinstance(candidate, str):
        text_id = canonical_taxonomy_id(candidate)
        if text_id in taxonomy_by_id:
            return taxonomy_by_id[text_id]
        if normalize(candidate) in taxonomy_by_name:
            return taxonomy_by_name[normalize(candidate)]

    return None

FIT_SYSTEM_PROMPT = r'''
You are a strict verifier of an already-assigned AI-harm taxonomy class.

You are NOT a classifier.

You MUST NOT choose, suggest, infer, rewrite, normalize, or replace a class.

You receive exactly ONE candidate class already assigned upstream.
Your only task is to decide whether THIS class fits THIS extracted event.

Use the authoritative Short definition and Inclusion test (proximate cause)
as the main criteria.

Return:
- "fit": event substantively satisfies this exact class.
- "not_fit": event clearly does not satisfy this exact class.
- "uncertain": evidence is insufficient for a safe decision.

Do not mark fit merely because the class is broadly related to AI harm.

Return JSON ONLY:
{
  "status": "fit",
  "confidence": 0.0,
  "reason": "brief evidence-grounded reason"
}

status must be exactly fit, not_fit, or uncertain.
reason must be at most 35 words.
'''

def verifier_event_view(event):
    return {
        "event_type": event.get("event_type", ""),
        "ai_system": event.get("ai_system", ""),
        "organization": event.get("organization", ""),
        "affected_group": event.get("affected_group", ""),
        "action": event.get("action", ""),
        "consequence": event.get("consequence", ""),
        "location": event.get("location", ""),
        "event_date": event.get("event_date", ""),
        "original_evidence_span": event.get("original_evidence_span", ""),
        "translated_evidence_span": event.get("translated_evidence_span", ""),
    }

def build_fit_prompt(event, candidate, row):
    payload = {
        "event": verifier_event_view(event),
        "candidate_classification": candidate,
        "authoritative_taxonomy_row": {
            "#": row.get("#", ""),
            "Branch": row.get("Branch", ""),
            "Subcategory": row.get("Subcategory", ""),
            "Short definition": row.get("Short definition", ""),
            "Inclusion test (proximate cause)": row.get(
                "Inclusion test (proximate cause)", ""
            ),
            "Protected interest": row.get("Protected interest", ""),
            "Event type": row.get("Event type", ""),
            "Occurrence status admitted": row.get(
                "Occurrence status admitted", ""
            ),
        },
    }
    return (
        "Does THIS supplied class fit THIS event? Evaluate only that candidate.\n\n"
        + json.dumps(payload, ensure_ascii=False, indent=2)
    )

verification_jobs = []

for event in combined_events:
    for idx, candidate in enumerate(
        classification_candidates(event.get("harm_category"))
    ):
        verification_jobs.append({
            "job_id": f"{clean(event['report_id'])}::{idx}",
            "report_id": clean(event["report_id"]),
            "candidate_index": idx,
            "candidate": candidate,
            "taxonomy_row": match_taxonomy_row(candidate),
            "event": event,
        })

print("Combined events:", len(combined_events))
print("Candidate categories:", len(verification_jobs))
print(
    "Matched to Excel:",
    sum(j["taxonomy_row"] is not None for j in verification_jobs),
)


## 5. Field-level extraction evaluation

In [ ]:
FIELDS = [
    "event_type",
    "ai_system",
    "organization",
    "affected_group",
    "action",
    "harm_category",
    "consequence",
    "location",
    "event_date",
]

def token_f1(prediction, reference):
    pred_tokens = normalize(prediction).split()
    ref_tokens = normalize(reference).split()

    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1
    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    overlap = sum(min(count, ref_counts.get(token, 0)) for token, count in pred_counts.items())
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def date_score(prediction, reference):
    pred = clean(prediction)
    ref = clean(reference)

    if pred == ref:
        return 1.0
    if not pred or not ref:
        return 0.0

    pred_year = year_from(pred)
    ref_year = year_from(ref)

    return 0.5 if pred_year and pred_year == ref_year else 0.0

def evaluate_extraction(predictions, gold_events, reports_by_id):
    prediction_by_id = {row["report_id"]: row for row in predictions}
    rows = []

    for gold in gold_events:
        report_id = gold["report_id"]
        pred = prediction_by_id.get(report_id, {})

        for field in FIELDS:
            predicted = clean(pred.get(field))
            reference = clean(gold.get(field))
            exact = float(normalize(predicted) == normalize(reference))
            f1 = token_f1(predicted, reference)

            if field == "event_date":
                exact = date_score(predicted, reference)

            rows.append({
                "report_id": report_id,
                "field": field,
                "prediction": predicted,
                "gold": reference,
                "exact": exact,
                "token_f1": f1,
            })

    details = pd.DataFrame(rows)
    summary = details.groupby("field", as_index=False).agg(
        exact_match=("exact", "mean"),
        token_f1=("token_f1", "mean"),
    )

    return details, summary

extraction_details_df, extraction_summary_df = evaluate_extraction(
    predictions,
    gold_events_eval,
    reports_by_id,
)

display(extraction_summary_df)
display(extraction_details_df.head(20))

extraction_summary_df.to_csv(OUTPUT_DIR / "extraction_summary.csv", index=False)
extraction_details_df.to_csv(OUTPUT_DIR / "extraction_details.csv", index=False)

# Verify deterministic source-evidence attachment separately from model-field accuracy.
evidence_integrity_rows = []
for prediction in predictions:
    report_id = prediction["report_id"]
    report = reports_by_id[report_id]
    original_text = clean(report.get("original_text"))
    translated_text = clean(report.get("translated_text"))
    source_language = clean(report.get("source_language")).lower()

    expected_original = original_text or "Not specified in report"
    expected_translation = (
        original_text if source_language == "en" and original_text
        else translated_text if source_language != "en" and translated_text
        else "Not specified in report"
    )

    evidence_integrity_rows.append({
        "report_id": report_id,
        "original_exact": prediction.get("original_evidence_span") == expected_original,
        "translated_exact": prediction.get("translated_evidence_span") == expected_translation,
    })

evidence_integrity_df = pd.DataFrame(evidence_integrity_rows)
if not evidence_integrity_df.empty:
    assert evidence_integrity_df["original_exact"].all()
    assert evidence_integrity_df["translated_exact"].all()

evidence_integrity_df.to_csv(OUTPUT_DIR / "evidence_integrity.csv", index=False)
display(evidence_integrity_df.head(20))


## 6. Prepare temporal hyperedges

A complete incident is represented as one hyperedge containing:

\[
\{organization,\ ai\ system,\ affected\ group,\ action,\ harm,\ consequence,\ location,\ source\}
\]

The timestamp and provenance score are attached to the hyperedge.

In [ ]:
ROLES = [
    "organization",
    "ai_system",
    "affected_group",
    "action",
    "harm_category",
    "consequence",
    "location",
]

EMPTY = "__EMPTY__"

def prediction_provenance(prediction):
    """Inference-time provenance must not use gold labels.

    WIP single-pass proxy: use the extractor's self-reported confidence, clipped to [0,1].
    Missing/non-numeric confidence defaults to 0.5.
    """
    try:
        value = float(prediction.get("confidence", 0.5))
    except Exception:
        value = 0.5
    return float(np.clip(value, 0.0, 1.0))

def build_hyperedges(events, use_gold_provenance=False):
    edges = []

    for event in events:
        report_id = clean(event.get("report_id"))
        if not report_id or report_id not in reports_by_id:
            continue

        report = reports_by_id[report_id]
        provenance = 1.0 if use_gold_provenance else prediction_provenance(event)

        edges.append({
            "event_id": report_id,
            "values": {
                "organization": clean(event.get("organization")) or EMPTY,
                "ai_system": clean(event.get("ai_system")) or EMPTY,
                "affected_group": clean(event.get("affected_group")) or EMPTY,
                "action": clean(event.get("action")) or EMPTY,
                "harm_category": graph_text(event.get("harm_category")) or EMPTY,
                "consequence": clean(event.get("consequence")) or EMPTY,
                "location": clean(event.get("location")) or clean(report.get("country")) or EMPTY,
            },
            "year": year_from(event.get("event_date"), report.get("publication_date")),
            "provenance": provenance,
        })

    return edges

# Gold graph is evaluation/reference only and uses the overlap with current input.
gold_hyperedges = build_hyperedges(gold_events_eval, use_gold_provenance=True)

# Predicted graph contains ALL successfully inferred input records.
predicted_hyperedges = build_hyperedges(predictions, use_gold_provenance=False)

print("Gold-overlap hyperedges:", len(gold_hyperedges))
print("Predicted inference hyperedges:", len(predicted_hyperedges))


## 7. Train-only graph preparation

In [ ]:
def prepare_graph(edges):
    if len(edges) < 8:
        train_edges = list(edges)
        random.Random(SEED).shuffle(train_edges)
        validation_edges = []
        test_edges = []
    else:
        dated = sorted(
            [edge for edge in edges if edge["year"] > 0],
            key=lambda edge: (edge["year"], edge["event_id"]),
        )
        undated = [edge for edge in edges if edge["year"] == 0]

        if len(dated) < 8:
            dated = list(edges)
            random.Random(SEED).shuffle(dated)
            undated = []

        test_n = max(1, round(len(dated) * TEST_FRACTION))
        val_n = max(1, round(len(dated) * VALIDATION_FRACTION))
        train_end = len(dated) - val_n - test_n
        train_edges = dated[:train_end] + undated
        validation_edges = dated[train_end:train_end + val_n]
        test_edges = dated[train_end + val_n:]

    vocab = {}
    offsets = {}
    current = 0

    for role in ROLES:
        values = sorted({normalize(edge["values"][role]) or EMPTY for edge in train_edges})
        role_vocab = {"__UNK__": 0, EMPTY: 1}

        for value in values:
            if value not in role_vocab:
                role_vocab[value] = len(role_vocab)

        vocab[role] = role_vocab
        offsets[role] = current
        current += len(role_vocab)

    years = [edge["year"] for edge in train_edges if edge["year"] > 0]
    year_min = min(years) if years else 2000
    year_max = max(years) if years else year_min + 1

    def encode(items):
        rows = []

        for edge in items:
            ids = []

            for role in ROLES:
                local_id = vocab[role].get(normalize(edge["values"][role]) or EMPTY, 0)
                ids.append(offsets[role] + local_id)

            known = float(edge["year"] > 0)
            year = (edge["year"] - year_min) / max(1, year_max - year_min) if edge["year"] > 0 else 0.0

            rows.append({
                "event_id": edge["event_id"],
                "global_ids": ids,
                "year": float(year),
                "known": known,
                "raw_year": edge["year"],
                "provenance": float(edge["provenance"]),
            })

        return rows

    train_rows = encode(train_edges)

    # ---- Chronological year buckets, for the recurrent/temporal-evolution training path. ----
    # Bucket 0 (key None) holds undated rows and is always processed first as a warm-up:
    # it lets undated entities register in entity_state before any chronological rollout
    # begins, without claiming a false position in the timeline.
    buckets = {}
    for row in train_rows:
        key = row["raw_year"] if row["raw_year"] > 0 else None
        buckets.setdefault(key, []).append(row)

    ordered_keys = sorted([k for k in buckets if k is not None])
    train_by_year = []
    if None in buckets:
        train_by_year.append((None, buckets[None]))
    for key in ordered_keys:
        train_by_year.append((key, buckets[key]))

    return {
        "train": train_rows,
        "train_by_year": train_by_year,
        "validation": encode(validation_edges),
        "test": encode(test_edges),
        "vocab": vocab,
        "offsets": offsets,
        "total_nodes": current,
        "year_min": year_min,
        "year_max": year_max,
    }

def to_tensors(rows):
    return (
        torch.tensor([row["global_ids"] for row in rows], dtype=torch.long, device=DEVICE),
        torch.tensor([row["year"] for row in rows], dtype=torch.float32, device=DEVICE),
        torch.tensor([row["known"] for row in rows], dtype=torch.float32, device=DEVICE),
        torch.tensor([row["provenance"] for row in rows], dtype=torch.float32, device=DEVICE),
    )


## 8. PHTKG

The model performs:

\[
H^{(l+1)} = f(H^{(l)}, \mathcal{E}, T, P)
\]

- Hyperedges aggregate all incident roles through attention.
- Time enters the event update as a trainable continuous message.
- Provenance modifies event and entity propagation.
- Event states propagate back to entity nodes through learned temporal decay.
- Recurring relational-temporal structures become similar in the learned embedding space.

In [ ]:
def current_phtkg_config():
    """Return the currently active PHTKG hyperparameters as a plain dict."""
    return {
        "EMBEDDING_DIM": int(EMBEDDING_DIM),
        "LAYERS": int(LAYERS),
        "EPOCHS": int(EPOCHS),
        "LEARNING_RATE": float(LEARNING_RATE),
        "WEIGHT_DECAY": float(WEIGHT_DECAY),
        "DROPOUT": float(DROPOUT),
        "LABEL_SMOOTHING": float(LABEL_SMOOTHING),
        "NEGATIVES": int(NEGATIVES),
        "PATIENCE": int(PATIENCE),
        "EXTRAPOLATION_WEIGHT": float(EXTRAPOLATION_WEIGHT),
        "EXTRAPOLATION_WARMUP_EPOCHS": int(EXTRAPOLATION_WARMUP_EPOCHS),
        "RANKING_MARGIN": float(RANKING_MARGIN),
        "RANKING_WEIGHT": float(RANKING_WEIGHT),
        "GRAD_CLIP": float(GRAD_CLIP),
        "HARD_NEGATIVE_POOL_MULTIPLIER": int(HARD_NEGATIVE_POOL_MULTIPLIER),
        "LR_SCHEDULER_PATIENCE": int(LR_SCHEDULER_PATIENCE),
    }

def apply_phtkg_config(config):
    """Apply a selected Auto-PHTKG configuration to notebook-level defaults.

    This keeps the later ablation/diagnostic cells aligned with the final model.
    """
    global EMBEDDING_DIM, LAYERS, EPOCHS, LEARNING_RATE, WEIGHT_DECAY, DROPOUT
    global LABEL_SMOOTHING, NEGATIVES, PATIENCE, EXTRAPOLATION_WEIGHT
    global EXTRAPOLATION_WARMUP_EPOCHS, RANKING_MARGIN, RANKING_WEIGHT
    global GRAD_CLIP, HARD_NEGATIVE_POOL_MULTIPLIER, LR_SCHEDULER_PATIENCE

    for key, value in config.items():
        if key == "CORRUPTION_WEIGHTS":
            continue
        if key in globals():
            globals()[key] = value

class PHTKG(nn.Module):
    """Provenance-aware temporal hypergraph model with configuration injection.

    `config=None` preserves the original notebook behavior.  Auto-PHTKG passes a
    trial-specific configuration, allowing dimensions/layers/dropout to vary without
    mutating architecture code.
    """

    def __init__(self, total_nodes, config=None):
        super().__init__()
        cfg = current_phtkg_config()
        if config:
            cfg.update(config)
        self.config = copy.deepcopy(cfg) if 'copy' in globals() else dict(cfg)

        d = int(cfg["EMBEDDING_DIM"])
        self.layers = int(cfg["LAYERS"])
        self.dimension = d
        self.entity_embeddings = nn.Embedding(total_nodes, d)
        self.role_embeddings = nn.Parameter(torch.randn(len(ROLES), d) * 0.02)
        self.event_seed = nn.Parameter(torch.randn(d) * 0.02)

        self.time_linear_weight = nn.Parameter(torch.randn(d))
        self.time_linear_bias = nn.Parameter(torch.zeros(d))
        self.time_periodic_weight = nn.Parameter(torch.randn(d))
        self.time_periodic_bias = nn.Parameter(torch.zeros(d))
        self.missing_time = nn.Parameter(torch.randn(d) * 0.02)

        self.provenance_message = nn.Sequential(nn.Linear(1, d), nn.Tanh())
        self.entity_key = nn.Linear(d, d, bias=False)
        self.entity_value = nn.Linear(d, d, bias=False)
        self.event_query = nn.Linear(d, d, bias=False)
        self.role_bias = nn.Parameter(torch.zeros(len(ROLES)))

        self.dropout = nn.Dropout(float(cfg["DROPOUT"]))
        self.event_update = nn.GRUCell(d, d)
        self.entity_update = nn.GRUCell(d, d)
        self.event_to_entity = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in ROLES])
        self.temporal_decay = nn.Parameter(torch.zeros(len(ROLES)))
        self.scorer = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.next_year_head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, 1))

    def encode_time(self, year_normalized, device):
        y = torch.tensor([[year_normalized]], dtype=torch.float32, device=device)
        vector = y * self.time_linear_weight + self.time_linear_bias + torch.sin(
            y * self.time_periodic_weight + self.time_periodic_bias
        )
        return vector.squeeze(0)

    def forward(self, global_ids, years, known, provenance, time_mean, time_known, entity_state_init=None):
        event_count = global_ids.shape[0]
        entity_state = self.entity_embeddings.weight if entity_state_init is None else entity_state_init
        event_state = self.event_seed.unsqueeze(0).expand(event_count, -1)

        years_column = years.unsqueeze(-1)
        known_column = known.unsqueeze(-1)
        observed_time = (
            years_column * self.time_linear_weight
            + self.time_linear_bias
            + torch.sin(years_column * self.time_periodic_weight + self.time_periodic_bias)
        )
        time_message = known_column * observed_time + (1 - known_column) * self.missing_time
        provenance_message = self.provenance_message(provenance.unsqueeze(-1))
        provenance_gate = torch.sigmoid(provenance).unsqueeze(-1)

        for _ in range(self.layers):
            incident = entity_state[global_ids] + self.role_embeddings.unsqueeze(0)
            keys = self.entity_key(incident)
            values = self.entity_value(incident)
            query = self.event_query(event_state).unsqueeze(1)

            logits = (query * keys).sum(-1) / math.sqrt(self.dimension) + self.role_bias.unsqueeze(0)
            attention = F.softmax(logits, dim=1)
            entity_message = self.dropout((attention.unsqueeze(-1) * values).sum(1))
            event_state = self.event_update(
                entity_message + time_message + provenance_message,
                event_state,
            )

            aggregate = torch.zeros_like(entity_state)
            denominator = torch.zeros(entity_state.shape[0], 1, device=entity_state.device)

            for role_index in range(len(ROLES)):
                ids = global_ids[:, role_index]
                temporal_known = known * time_known[ids]
                gap = torch.abs(years - time_mean[ids])
                decay = F.softplus(self.temporal_decay[role_index])
                temporal_weight = temporal_known * torch.exp(-decay * gap) + (1 - temporal_known)
                weight = temporal_weight * provenance_gate.squeeze(-1)
                message = self.event_to_entity[role_index](event_state) * weight.unsqueeze(-1)

                aggregate.index_add_(0, ids, message)
                denominator.index_add_(0, ids, weight.unsqueeze(-1))

            # Incident-only evolution: untouched nodes preserve their previous state.
            entity_input = self.dropout(aggregate / torch.clamp(denominator, min=1.0))
            proposed_state = self.entity_update(entity_input, entity_state)
            active_mask = denominator > 0
            entity_state = torch.where(active_mask, proposed_state, entity_state)

        embedding = F.normalize(event_state, dim=-1)
        return {
            "embedding": embedding,
            "score": self.scorer(embedding).squeeze(-1),
            "entity_state": entity_state,
        }

    def predict_recurrence(self, entity_ids, entity_state, target_year_normalized):
        time_query = self.encode_time(target_year_normalized, entity_state.device)
        time_query = time_query.unsqueeze(0).expand(len(entity_ids), -1)
        logits = self.next_year_head(
            torch.cat([entity_state[entity_ids], time_query], dim=-1)
        ).squeeze(-1)
        return torch.sigmoid(logits)


## 9. Training and chronological evaluation

In [ ]:
def graph_context(rows, total_nodes):
    global_ids, years, known, provenance = to_tensors(rows)
    sums = torch.zeros(total_nodes, device=DEVICE)
    counts = torch.zeros(total_nodes, device=DEVICE)

    for role_index in range(len(ROLES)):
        ids = global_ids[:, role_index]
        sums.index_add_(0, ids, years * known)
        counts.index_add_(0, ids, known)

    return (
        global_ids, years, known, provenance,
        sums / torch.clamp(counts, min=1.0),
        (counts > 0).float(),
    )

def binary_auc(labels, scores):
    positives = scores[labels == 1]
    negatives = scores[labels == 0]
    if not len(positives) or not len(negatives):
        return float("nan")
    wins = 0.0
    total = 0
    for positive in positives:
        for negative in negatives:
            total += 1
            wins += 1.0 if positive > negative else 0.5 if positive == negative else 0.0
    return wins / total

EVAL_NEGATIVE_DRAWS = 5

def _set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@contextmanager
def _preserve_rng(seed):
    """Deterministic validation corruptions without perturbing training RNG state."""
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    _set_all_seeds(seed)
    try:
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)

def evaluate(model, rows, time_mean, time_known, graph, entity_state_init=None, seed=SEED + 404):
    if not rows:
        return {"count": 0, "auc": float("nan"), "auc_std": float("nan"),
                "positive_mean": float("nan"), "negative_mean": float("nan")}

    global_ids, years, known, provenance = to_tensors(rows)
    model.eval()
    with _preserve_rng(seed), torch.inference_mode():
        positive = torch.sigmoid(
            model(global_ids, years, known, provenance, time_mean, time_known,
                  entity_state_init=entity_state_init)["score"]
        ).cpu().numpy()

        aucs, negative_means = [], []
        for _ in range(EVAL_NEGATIVE_DRAWS):
            negative_ids = corrupt(global_ids, graph["vocab"], graph["offsets"], 3)
            negative = torch.sigmoid(
                model(negative_ids, years.repeat_interleave(3), known.repeat_interleave(3),
                      provenance.repeat_interleave(3), time_mean, time_known,
                      entity_state_init=entity_state_init)["score"]
            ).cpu().numpy()
            labels = np.concatenate([np.ones(len(positive)), np.zeros(len(negative))])
            scores = np.concatenate([positive, negative])
            aucs.append(binary_auc(labels, scores))
            negative_means.append(float(negative.mean()))

    return {
        "count": len(rows),
        "auc": float(np.mean(aucs)),
        "auc_std": float(np.std(aucs)),
        "positive_mean": float(positive.mean()),
        "negative_mean": float(np.mean(negative_means)),
    }

def paired_evaluate_for_selection(
    model, rows, time_mean, time_known, graph, entity_state_init=None,
    copies=3, draws=5, seed=SEED + 505,
):
    """Leakage-safe model-selection metric: real event vs its own corruptions.

    Raw logits are used, so saturation of sigmoid probabilities cannot hide margins.
    """
    if not rows:
        return {"count": 0, "paired_auc": float("nan"), "paired_auc_std": float("nan")}

    global_ids, years, known, provenance = to_tensors(rows)
    model.eval()
    draw_scores = []
    with _preserve_rng(seed), torch.inference_mode():
        positive = model(
            global_ids, years, known, provenance, time_mean, time_known,
            entity_state_init=entity_state_init,
        )["score"].cpu().numpy()

        for _ in range(draws):
            negative_ids = corrupt(global_ids, graph["vocab"], graph["offsets"], copies)
            negative = model(
                negative_ids, years.repeat_interleave(copies), known.repeat_interleave(copies),
                provenance.repeat_interleave(copies), time_mean, time_known,
                entity_state_init=entity_state_init,
            )["score"].cpu().numpy()
            wins, total = 0.0, 0
            for i in range(len(positive)):
                own = negative[i * copies:(i + 1) * copies]
                for neg in own:
                    total += 1
                    wins += 1.0 if positive[i] > neg else 0.5 if positive[i] == neg else 0.0
            draw_scores.append(wins / total if total else float("nan"))

    return {
        "count": len(rows),
        "paired_auc": float(np.nanmean(draw_scores)),
        "paired_auc_std": float(np.nanstd(draw_scores)),
    }

def sample_negative_entities(positive_ids, total_nodes, count):
    positive_set = set(positive_ids)
    pool = [i for i in range(total_nodes) if i not in positive_set]
    if not pool:
        return []
    return random.sample(pool, min(count, len(pool)))

def _rollout_real_entity_state(model, graph, time_mean, time_known):
    buckets = graph["train_by_year"]
    chronological = sum(1 for key, _ in buckets if key is not None) >= 2
    model.eval()
    with torch.no_grad():
        if not chronological:
            return model.entity_embeddings.weight.detach().clone(), []

        entity_state = model.entity_embeddings.weight
        drift = []
        for bucket_year, bucket_rows in buckets:
            b_ids, b_years, b_known, b_prov = to_tensors(bucket_rows)
            out = model(
                b_ids, b_years, b_known, b_prov, time_mean, time_known,
                entity_state_init=entity_state,
            )
            new_state = out["entity_state"]
            if bucket_year is not None:
                drift.append({
                    "year": bucket_year,
                    "drift": float(torch.norm(new_state - entity_state, dim=-1).mean().item()),
                })
            entity_state = new_state
    return entity_state.detach().clone(), drift

def _hard_negative_ids(
    model, global_ids, years, known, provenance, time_mean, time_known,
    graph, copies, pool_multiplier, entity_state_init=None,
):
    """Generate a larger legal corruption pool and keep the highest-scoring negatives."""
    pool_multiplier = max(1, int(pool_multiplier))
    pool_copies = int(copies) * pool_multiplier
    candidates = corrupt(global_ids, graph["vocab"], graph["offsets"], pool_copies)
    if pool_multiplier == 1:
        return candidates

    with torch.no_grad():
        candidate_scores = model(
            candidates,
            years.repeat_interleave(pool_copies),
            known.repeat_interleave(pool_copies),
            provenance.repeat_interleave(pool_copies),
            time_mean, time_known,
            entity_state_init=entity_state_init,
        )["score"].view(len(global_ids), pool_copies)
        top_idx = candidate_scores.topk(k=int(copies), dim=1).indices

    candidate_view = candidates.view(len(global_ids), pool_copies, global_ids.shape[1])
    batch_idx = torch.arange(len(global_ids), device=global_ids.device).unsqueeze(1)
    selected = candidate_view[batch_idx, top_idx]
    return selected.reshape(-1, global_ids.shape[1])

def train_phtkg(
    graph, label, config=None, max_epochs=None, patience_override=None,
    evaluate_test=True, verbose=True, seed=SEED,
):
    global CURRENT_REAL_SIGNATURES, CURRENT_CORRUPTION_WEIGHTS

    cfg = current_phtkg_config()
    if config:
        cfg.update(config)
    epochs = int(max_epochs if max_epochs is not None else cfg["EPOCHS"])
    patience_limit = int(patience_override if patience_override is not None else cfg["PATIENCE"])
    _set_all_seeds(seed)

    train_rows = graph["train"]
    if len(train_rows) < 4:
        raise RuntimeError(f"{label}: at least four training events are required.")

    global_ids, years, known, provenance, time_mean, time_known = graph_context(
        train_rows, graph["total_nodes"]
    )
    buckets = graph["train_by_year"]
    chronological = sum(1 for key, _ in buckets if key is not None) >= 2
    if not chronological and verbose:
        print(f"{label}: fewer than two distinct years -- flat training fallback.")

    if "CORRUPTION_WEIGHTS" in cfg:
        CURRENT_CORRUPTION_WEIGHTS = dict(cfg["CORRUPTION_WEIGHTS"])
    else:
        CURRENT_CORRUPTION_WEIGHTS = dict(CORRUPTION_WEIGHTS)

    model = PHTKG(graph["total_nodes"], config=cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(cfg["LEARNING_RATE"]), weight_decay=float(cfg["WEIGHT_DECAY"])
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5,
        patience=int(cfg.get("LR_SCHEDULER_PATIENCE", 6)), min_lr=1e-6,
    )

    train_signatures = build_real_signatures(graph, splits=("train",))
    train_val_signatures = build_real_signatures(graph, splits=("train", "validation"))
    all_signatures = (
        build_real_signatures(graph, splits=("train", "validation", "test"))
        if evaluate_test else None
    )

    history = []
    best_selection = -1.0
    best_state = None
    best_final_entity_state = None
    best_drift = None
    patience_count = 0

    neg_copies = int(cfg["NEGATIVES"])
    hard_pool = int(cfg.get("HARD_NEGATIVE_POOL_MULTIPLIER", 1))
    label_smoothing = float(cfg["LABEL_SMOOTHING"])

    for epoch in range(1, epochs + 1):
        CURRENT_REAL_SIGNATURES = train_signatures
        model.train()
        optimizer.zero_grad(set_to_none=True)

        if not chronological:
            negative_ids = _hard_negative_ids(
                model, global_ids, years, known, provenance, time_mean, time_known,
                graph, neg_copies, hard_pool, entity_state_init=None,
            )
            positive_scores = model(global_ids, years, known, provenance, time_mean, time_known)["score"]
            negative_scores = model(
                negative_ids, years.repeat_interleave(neg_copies),
                known.repeat_interleave(neg_copies), provenance.repeat_interleave(neg_copies),
                time_mean, time_known,
            )["score"]

            bce_loss = (
                F.binary_cross_entropy_with_logits(
                    positive_scores, torch.full_like(positive_scores, 1.0 - label_smoothing)
                )
                + F.binary_cross_entropy_with_logits(
                    negative_scores, torch.full_like(negative_scores, label_smoothing)
                )
            ) / 2
            ranking_loss = F.relu(
                float(cfg["RANKING_MARGIN"])
                - positive_scores.repeat_interleave(neg_copies) + negative_scores
            ).mean()
            prediction_loss = bce_loss + float(cfg["RANKING_WEIGHT"]) * ranking_loss
            extrapolation_loss = torch.tensor(0.0, device=DEVICE)
        else:
            entity_state = model.entity_embeddings.weight
            prediction_losses = []
            extrapolation_losses = []

            for bucket_index, (bucket_year, bucket_rows) in enumerate(buckets):
                b_ids, b_years, b_known, b_prov = to_tensors(bucket_rows)
                negative_ids = _hard_negative_ids(
                    model, b_ids, b_years, b_known, b_prov, time_mean, time_known,
                    graph, neg_copies, hard_pool, entity_state_init=entity_state,
                )

                positive_out = model(
                    b_ids, b_years, b_known, b_prov, time_mean, time_known,
                    entity_state_init=entity_state,
                )
                negative_out = model(
                    negative_ids, b_years.repeat_interleave(neg_copies),
                    b_known.repeat_interleave(neg_copies), b_prov.repeat_interleave(neg_copies),
                    time_mean, time_known, entity_state_init=entity_state,
                )

                bucket_bce = (
                    F.binary_cross_entropy_with_logits(
                        positive_out["score"], torch.full_like(positive_out["score"], 1.0 - label_smoothing)
                    )
                    + F.binary_cross_entropy_with_logits(
                        negative_out["score"], torch.full_like(negative_out["score"], label_smoothing)
                    )
                ) / 2
                ranking_loss = F.relu(
                    float(cfg["RANKING_MARGIN"])
                    - positive_out["score"].repeat_interleave(neg_copies)
                    + negative_out["score"]
                ).mean()
                prediction_losses.append(bucket_bce + float(cfg["RANKING_WEIGHT"]) * ranking_loss)

                new_entity_state = positive_out["entity_state"]

                if bucket_index + 1 < len(buckets):
                    next_year, next_rows = buckets[bucket_index + 1]
                    if next_year is not None:
                        positive_ids = sorted({gid for row in next_rows for gid in row["global_ids"]})
                        if positive_ids:
                            negative_entity_ids = sample_negative_entities(
                                positive_ids, graph["total_nodes"], len(positive_ids)
                            )
                            next_norm = (next_year - graph["year_min"]) / max(
                                1, graph["year_max"] - graph["year_min"]
                            )
                            pos_tensor = torch.tensor(positive_ids, device=DEVICE)
                            pos_logits = model.next_year_head(torch.cat([
                                new_entity_state[pos_tensor],
                                model.encode_time(next_norm, DEVICE).unsqueeze(0).expand(len(pos_tensor), -1),
                            ], dim=-1)).squeeze(-1)
                            transition_loss = F.binary_cross_entropy_with_logits(
                                pos_logits, torch.ones_like(pos_logits)
                            )
                            if negative_entity_ids:
                                neg_tensor = torch.tensor(negative_entity_ids, device=DEVICE)
                                neg_logits = model.next_year_head(torch.cat([
                                    new_entity_state[neg_tensor],
                                    model.encode_time(next_norm, DEVICE).unsqueeze(0).expand(len(neg_tensor), -1),
                                ], dim=-1)).squeeze(-1)
                                transition_loss = (transition_loss + F.binary_cross_entropy_with_logits(
                                    neg_logits, torch.zeros_like(neg_logits)
                                )) / 2
                            extrapolation_losses.append(transition_loss)

                entity_state = new_entity_state

            prediction_loss = torch.stack(prediction_losses).mean()
            extrapolation_loss = (
                torch.stack(extrapolation_losses).mean()
                if extrapolation_losses else torch.tensor(0.0, device=DEVICE)
            )

        warmup = max(1, int(cfg["EXTRAPOLATION_WARMUP_EPOCHS"]))
        extrapolation_weight = float(cfg["EXTRAPOLATION_WEIGHT"]) * min(1.0, epoch / warmup)
        loss = prediction_loss + extrapolation_weight * extrapolation_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.get("GRAD_CLIP", 1.5)))
        optimizer.step()

        synced_final_entity_state, synced_drift = _rollout_real_entity_state(
            model, graph, time_mean, time_known
        )

        CURRENT_REAL_SIGNATURES = train_val_signatures
        val_rows = graph["validation"] or graph["train"]
        validation = evaluate(
            model, val_rows, time_mean, time_known, graph,
            entity_state_init=synced_final_entity_state, seed=seed + 100_000,
        )
        paired = paired_evaluate_for_selection(
            model, val_rows, time_mean, time_known, graph,
            entity_state_init=synced_final_entity_state, seed=seed + 200_000,
        )

        selection_score = paired["paired_auc"]
        history.append({
            "epoch": epoch,
            "loss": float(loss.item()),
            "prediction_loss": float(prediction_loss.item()),
            "extrapolation_loss": float(extrapolation_loss.item()),
            "validation_auc": validation["auc"],
            "validation_paired_auc": paired["paired_auc"],
            "validation_paired_auc_std": paired["paired_auc_std"],
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
        })

        scheduler.step(selection_score if not math.isnan(selection_score) else 0.0)

        if not math.isnan(selection_score) and selection_score > best_selection + 1e-4:
            best_selection = selection_score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_final_entity_state = synced_final_entity_state.detach().cpu().clone()
            best_drift = synced_drift
            patience_count = 0
        else:
            patience_count += 1

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                f"{label} | epoch {epoch:03d} | loss={loss.item():.4f} | "
                f"val-AUC={validation['auc']:.4f} | paired={selection_score:.4f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )
        if patience_count >= patience_limit:
            if verbose:
                print(f"{label} | early stopping at epoch {epoch}")
            break

    if best_state:
        model.load_state_dict(best_state)

    restored_final_state, restored_drift = _rollout_real_entity_state(
        model, graph, time_mean, time_known
    )
    best_final_entity_state = restored_final_state.detach().cpu().clone()
    best_drift = restored_drift
    final_state_device = restored_final_state.to(DEVICE)

    CURRENT_REAL_SIGNATURES = train_signatures
    train_metric = evaluate(model, graph["train"], time_mean, time_known, graph,
                            entity_state_init=final_state_device, seed=seed + 300_000)
    train_paired = paired_evaluate_for_selection(
        model, graph["train"], time_mean, time_known, graph,
        entity_state_init=final_state_device, seed=seed + 310_000,
    )

    CURRENT_REAL_SIGNATURES = train_val_signatures
    validation_metric = evaluate(model, graph["validation"], time_mean, time_known, graph,
                                 entity_state_init=final_state_device, seed=seed + 400_000)
    validation_paired = paired_evaluate_for_selection(
        model, graph["validation"], time_mean, time_known, graph,
        entity_state_init=final_state_device, seed=seed + 410_000,
    )

    if evaluate_test:
        CURRENT_REAL_SIGNATURES = all_signatures
        test_metric = evaluate(model, graph["test"], time_mean, time_known, graph,
                               entity_state_init=final_state_device, seed=seed + 500_000)
        test_paired = paired_evaluate_for_selection(
            model, graph["test"], time_mean, time_known, graph,
            entity_state_init=final_state_device, seed=seed + 510_000,
        )
    else:
        test_metric = {"count": len(graph["test"]), "auc": float("nan"), "auc_std": float("nan"),
                       "positive_mean": float("nan"), "negative_mean": float("nan")}
        test_paired = {"paired_auc": float("nan"), "paired_auc_std": float("nan")}

    metrics = pd.DataFrame([
        {"experiment": label, "split": "train", **train_metric, **train_paired},
        {"experiment": label, "split": "validation", **validation_metric, **validation_paired},
        {"experiment": label, "split": "test", **test_metric, **test_paired},
    ])

    return (
        model, pd.DataFrame(history), metrics, time_mean, time_known,
        best_final_entity_state,
        pd.DataFrame(best_drift) if best_drift else pd.DataFrame(columns=["year", "drift"]),
    )


In [ ]:
DIRECTIONAL_ROLE_PAIRS = [("organization", "affected_group")]

CORRUPTION_WEIGHTS = {
    "same_role_substitution": 0.55,
    "role_swap": 0.25,
    "cross_event_borrow": 0.20,
}
CURRENT_CORRUPTION_WEIGHTS = dict(CORRUPTION_WEIGHTS)

def build_real_signatures(graph, splits=("train", "validation", "test")):
    """Return real hyperedge signatures for explicitly allowed splits."""
    rows = []
    for split in splits:
        rows.extend(graph.get(split, []))
    return {tuple(row["global_ids"]) for row in rows}

def rule_based_corrupt(global_ids, vocab, offsets, copies, real_signatures=None, weights=None):
    n_rows, n_roles = global_ids.shape
    device = global_ids.device
    weight_map = weights or CURRENT_CORRUPTION_WEIGHTS
    strategies = list(weight_map.keys())
    strategy_weights = list(weight_map.values())
    output_rows = []

    for row_index in range(n_rows):
        original = global_ids[row_index]
        for _ in range(copies):
            for attempt in range(8):
                ids = original.clone()
                strategy = random.choices(strategies, weights=strategy_weights)[0]

                if strategy == "role_swap" and DIRECTIONAL_ROLE_PAIRS:
                    role_a, role_b = random.choice(DIRECTIONAL_ROLE_PAIRS)
                    idx_a, idx_b = ROLES.index(role_a), ROLES.index(role_b)
                    ids[idx_a], ids[idx_b] = ids[idx_b].clone(), ids[idx_a].clone()
                else:
                    role_index = random.randrange(n_roles)
                    role = ROLES[role_index]
                    original_local = int(original[role_index]) - offsets[role]

                    if strategy == "cross_event_borrow":
                        column = global_ids[:, role_index]
                        candidates = column[column != original[role_index]]
                        if len(candidates) == 0:
                            continue
                        ids[role_index] = candidates[random.randrange(len(candidates))]
                    else:
                        vocab_size = len(vocab[role])
                        pool = [i for i in range(2, vocab_size) if i != original_local]
                        if not pool:
                            continue
                        ids[role_index] = offsets[role] + random.choice(pool)

                signature = tuple(int(x) for x in ids.tolist())
                if signature == tuple(int(x) for x in original.tolist()):
                    continue
                if real_signatures is not None and signature in real_signatures:
                    continue
                output_rows.append(ids)
                break
            else:
                # Safe fallback: cycle to another local id; guard against no-op/real positives.
                fallback = original.clone()
                made = False
                for role_index in range(n_roles):
                    role = ROLES[role_index]
                    vocab_size = len(vocab[role])
                    if vocab_size <= 2:
                        continue
                    original_local = int(original[role_index]) - offsets[role]
                    for local in range(2, vocab_size):
                        if local == original_local:
                            continue
                        fallback[role_index] = offsets[role] + local
                        signature = tuple(int(x) for x in fallback.tolist())
                        if real_signatures is None or signature not in real_signatures:
                            made = True
                            break
                    if made:
                        break
                if not made:
                    # Extremely small vocab: retain original only as last-resort shape safety.
                    fallback = original.clone()
                output_rows.append(fallback)

    return torch.stack(output_rows).to(device)

CURRENT_REAL_SIGNATURES = None

def corrupt(global_ids, vocab, offsets, copies):
    return rule_based_corrupt(
        global_ids, vocab, offsets, copies,
        real_signatures=CURRENT_REAL_SIGNATURES,
        weights=CURRENT_CORRUPTION_WEIGHTS,
    )


## 9b. Auto-PHTKG — self-parametric validation search

The tuner below **never evaluates the test split**.  It searches only against deterministic validation **paired AUC** (real hyperedge vs. its own legal corruptions), with a small penalty for instability across corruption draws.

It tunes capacity, optimizer/regularization, ranking and recurrence objectives, negative count, hard-negative mining intensity, and the corruption-strategy mixture.  The winning configuration is then frozen and used for both the gold-input and predicted-input experiments so their comparison remains fair.


In [ ]:
def _log_uniform(rng, low, high):
    return float(math.exp(rng.uniform(math.log(low), math.log(high))))

def _sample_corruption_weights(rng):
    # Dirichlet-like draw using Gamma variates; biased toward same-role negatives.
    raw = [rng.gammavariate(3.0, 1.0), rng.gammavariate(1.5, 1.0), rng.gammavariate(1.5, 1.0)]
    total = sum(raw)
    vals = [x / total for x in raw]
    return {
        "same_role_substitution": vals[0],
        "role_swap": vals[1],
        "cross_event_borrow": vals[2],
    }

def sample_auto_phtkg_config(rng, trial_index):
    # Trial 0 is always the current hand-tuned baseline.
    base = current_phtkg_config()
    base["CORRUPTION_WEIGHTS"] = dict(CORRUPTION_WEIGHTS)
    if trial_index == 0:
        return base

    base.update({
        "EMBEDDING_DIM": rng.choice([32, 48, 64, 96, 128]),
        "LAYERS": rng.choice([1, 2, 3]),
        "LEARNING_RATE": _log_uniform(rng, 1.5e-4, 3.0e-3),
        "WEIGHT_DECAY": _log_uniform(rng, 1e-6, 3e-3),
        "DROPOUT": rng.uniform(0.0, 0.35),
        "LABEL_SMOOTHING": rng.choice([0.0, 0.0, 0.02, 0.05]),
        "NEGATIVES": rng.randint(3, 10),
        "RANKING_MARGIN": rng.uniform(0.05, 0.60),
        "RANKING_WEIGHT": rng.uniform(0.15, 1.25),
        "EXTRAPOLATION_WEIGHT": rng.uniform(0.0, 0.30),
        "EXTRAPOLATION_WARMUP_EPOCHS": rng.choice([10, 20, 30, 40, 50]),
        "GRAD_CLIP": rng.choice([0.75, 1.0, 1.5, 2.0, 2.5]),
        "HARD_NEGATIVE_POOL_MULTIPLIER": rng.choice([1, 1, 2, 2, 3]),
        "LR_SCHEDULER_PATIENCE": rng.choice([4, 6, 8]),
        "CORRUPTION_WEIGHTS": _sample_corruption_weights(rng),
    })
    return base

def _flatten_trial_config(config):
    row = {k: v for k, v in config.items() if k != "CORRUPTION_WEIGHTS"}
    cw = config.get("CORRUPTION_WEIGHTS", CORRUPTION_WEIGHTS)
    row.update({
        "corrupt_same_role": cw["same_role_substitution"],
        "corrupt_role_swap": cw["role_swap"],
        "corrupt_cross_event": cw["cross_event_borrow"],
    })
    return row

def auto_tune_phtkg(graph, label="gold_events"):
    """Randomized, dependency-free hyperparameter optimization.

    Objective = validation paired-AUC - AUTO_TUNE_STD_PENALTY * paired-AUC std.
    The test split is not touched by train_phtkg(..., evaluate_test=False).
    """
    if not graph["validation"]:
        print("Auto-PHTKG skipped: no validation split is available.")
        cfg = current_phtkg_config()
        cfg["CORRUPTION_WEIGHTS"] = dict(CORRUPTION_WEIGHTS)
        return cfg, pd.DataFrame()

    rng = random.Random(AUTO_TUNE_SEED)
    records = []
    best_objective = -float("inf")
    best_config = None

    print(f"Auto-PHTKG: {AUTO_TUNE_TRIALS} leakage-safe trials on {label} validation data")
    print("Selection metric: paired AUC - stability penalty; TEST remains untouched.")

    for trial in range(AUTO_TUNE_TRIALS):
        cfg = sample_auto_phtkg_config(rng, trial)
        try:
            result = train_phtkg(
                graph, f"{label}_auto_trial_{trial + 1:02d}",
                config=cfg, max_epochs=AUTO_TUNE_EPOCHS,
                patience_override=AUTO_TUNE_PATIENCE, evaluate_test=False,
                verbose=False, seed=AUTO_TUNE_SEED + trial + 1,
            )
            _, hist, metrics, _, _, _, _ = result
            val = metrics.loc[metrics["split"] == "validation"].iloc[0]
            paired_auc = float(val["paired_auc"])
            paired_std = float(val["paired_auc_std"])
            pooled_auc = float(val["auc"])
            objective = paired_auc - AUTO_TUNE_STD_PENALTY * paired_std
            best_epoch = int(hist.loc[hist["validation_paired_auc"].idxmax(), "epoch"]) if len(hist) else 0
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                paired_auc = paired_std = pooled_auc = float("nan")
                objective = -float("inf")
                best_epoch = 0
                print(f"trial {trial + 1:02d}: OOM -> skipped")
            else:
                raise

        row = {
            "trial": trial + 1,
            "objective": objective,
            "validation_paired_auc": paired_auc,
            "validation_paired_auc_std": paired_std,
            "validation_auc": pooled_auc,
            "best_epoch": best_epoch,
            **_flatten_trial_config(cfg),
        }
        records.append(row)

        if objective > best_objective:
            best_objective = objective
            best_config = copy.deepcopy(cfg)

        print(
            f"trial {trial + 1:02d}/{AUTO_TUNE_TRIALS} | "
            f"paired={paired_auc:.4f} ± {paired_std:.4f} | objective={objective:.4f} | "
            f"d={cfg['EMBEDDING_DIM']} L={cfg['LAYERS']} neg={cfg['NEGATIVES']} "
            f"hardx={cfg['HARD_NEGATIVE_POOL_MULTIPLIER']}"
        )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    trials_df = pd.DataFrame(records).sort_values("objective", ascending=False).reset_index(drop=True)
    if best_config is None:
        best_config = current_phtkg_config()
        best_config["CORRUPTION_WEIGHTS"] = dict(CORRUPTION_WEIGHTS)

    print("\nBest Auto-PHTKG validation configuration:")
    print(json.dumps(best_config, indent=2))
    if len(trials_df):
        display(trials_df.head(10))
    return best_config, trials_df


## 10. Gold-input and predicted-input PHTKG experiments

In [ ]:
gold_graph = prepare_graph(gold_hyperedges)
predicted_graph = prepare_graph(predicted_hyperedges)

if AUTO_TUNE:
    best_phtkg_config, auto_tune_trials = auto_tune_phtkg(gold_graph, "gold_events")
    apply_phtkg_config(best_phtkg_config)
    CURRENT_CORRUPTION_WEIGHTS = dict(best_phtkg_config.get("CORRUPTION_WEIGHTS", CORRUPTION_WEIGHTS))
    auto_tune_trials.to_csv(OUTPUT_DIR / "auto_phtkg_trials.csv", index=False)
    (OUTPUT_DIR / "auto_phtkg_best_config.json").write_text(
        json.dumps(best_phtkg_config, indent=2), encoding="utf-8"
    )
else:
    best_phtkg_config = current_phtkg_config()
    best_phtkg_config["CORRUPTION_WEIGHTS"] = dict(CORRUPTION_WEIGHTS)
    auto_tune_trials = pd.DataFrame()

# Final fits: same selected configuration for a fair gold-vs-predicted comparison.
(gold_model, gold_history, gold_metrics, gold_time_mean, gold_time_known,
 gold_final_entity_state, gold_drift) = train_phtkg(
    gold_graph, "gold_events", config=best_phtkg_config,
    evaluate_test=True, verbose=True, seed=SEED,
)

(predicted_model, predicted_history, predicted_metrics, predicted_time_mean, predicted_time_known,
 predicted_final_entity_state, predicted_drift) = train_phtkg(
    predicted_graph, "predicted_events", config=best_phtkg_config,
    evaluate_test=True, verbose=True, seed=SEED + 1,
)

comparison_df = pd.concat([gold_metrics, predicted_metrics], ignore_index=True)
display(comparison_df)

gold_history.to_csv(OUTPUT_DIR / "gold_phtkg_training.csv", index=False)
predicted_history.to_csv(OUTPUT_DIR / "predicted_phtkg_training.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "phtkg_comparison.csv", index=False)


In [ ]:
def paired_evaluate(model, rows, time_mean, time_known, graph, entity_state_init=None, copies=3, draws=5):
    """Per-event AUC: does the real score beat only ITS OWN corrupted copies?
    Immune to any feature (like provenance level) that varies across events
    but is inherited unchanged by that event's own corrupted copies -- unlike
    evaluate()'s pooled AUC."""
    if not rows:
        return {"count": 0, "paired_auc": float("nan")}

    global_ids, years, known, provenance = to_tensors(rows)
    model.eval()

    with torch.inference_mode():
        positive = torch.sigmoid(
            model(global_ids, years, known, provenance, time_mean, time_known,
                  entity_state_init=entity_state_init)["score"]
        ).cpu().numpy()

        draw_scores = []
        for _ in range(draws):
            negative_ids = corrupt(global_ids, graph["vocab"], graph["offsets"], copies)
            negative = torch.sigmoid(
                model(negative_ids, years.repeat_interleave(copies), known.repeat_interleave(copies),
                      provenance.repeat_interleave(copies), time_mean, time_known,
                      entity_state_init=entity_state_init)["score"]
            ).cpu().numpy()

            wins, total = 0.0, 0
            for i in range(len(positive)):
                for neg in negative[i * copies:(i + 1) * copies]:
                    total += 1
                    wins += 1.0 if positive[i] > neg else 0.5 if positive[i] == neg else 0.0
            draw_scores.append(wins / total if total else float("nan"))

    return {"count": len(rows), "paired_auc": float(np.mean(draw_scores)), "paired_auc_std": float(np.std(draw_scores))}

CURRENT_REAL_SIGNATURES = build_real_signatures(gold_graph)  # FIX: use gold signatures here
print("gold_events paired test AUC:", paired_evaluate(
    gold_model, gold_graph["test"], gold_time_mean, gold_time_known, gold_graph,
    entity_state_init=gold_final_entity_state.to(DEVICE)))
CURRENT_REAL_SIGNATURES = build_real_signatures(predicted_graph)  # FIX: switch to predicted signatures
print("predicted_events paired test AUC:", paired_evaluate(
    predicted_model, predicted_graph["test"], predicted_time_mean, predicted_time_known, predicted_graph,
    entity_state_init=predicted_final_entity_state.to(DEVICE)))

## 10b. Temporal evolution — did the model actually learn it?

Two diagnostics, both produced *by training*, not computed afterward:

- **Entity-state drift per year** — `‖entity_state_t − entity_state_{t-1}‖`, logged at the best epoch. This is how much the chronological rollout moved entity representations from one year-bucket to the next. A flat line means the temporal-evolution objective isn't finding anything to learn (often just means too little data/years) — that's useful to know, not a bug to hide.
- **Recurrence-forecast quality** — how well `next_year_head` predicts, for each transition, which entities from year *t* reappear in year *t+1* vs. a negative sample of ones that don't. Reported as a simple AUC per transition. This is the actual "is this evolving/still active" predictor `predict_recurrence()` uses at inference time.

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
def recurrence_auc_per_transition(model, graph, time_mean, time_known, final_entity_state):
    """Re-run the chronological rollout once more (eval mode) to score how well
    next_year_head separates entities that DO recur next year from ones that don't."""
    buckets = graph["train_by_year"]
    entity_state = model.entity_embeddings.weight
    rows = []

    model.eval()
    with torch.inference_mode():
        for bucket_index, (bucket_year, bucket_rows) in enumerate(buckets):
            b_ids, b_years, b_known, b_prov = to_tensors(bucket_rows)
            out = model(b_ids, b_years, b_known, b_prov, time_mean, time_known, entity_state_init=entity_state)
            entity_state = out["entity_state"]

            if bucket_index + 1 < len(buckets):
                next_year, next_rows = buckets[bucket_index + 1]
                if next_year is None:
                    continue
                positive_ids = sorted({gid for row in next_rows for gid in row["global_ids"]})
                if not positive_ids:
                    continue
                negative_ids = sample_negative_entities(positive_ids, graph["total_nodes"], len(positive_ids))
                if not negative_ids:
                    continue

                next_norm = (next_year - graph["year_min"]) / max(1, graph["year_max"] - graph["year_min"])
                pos_prob = model.predict_recurrence(torch.tensor(positive_ids, device=DEVICE), entity_state, next_norm).cpu().numpy()
                neg_prob = model.predict_recurrence(torch.tensor(negative_ids, device=DEVICE), entity_state, next_norm).cpu().numpy()

                labels = np.concatenate([np.ones(len(pos_prob)), np.zeros(len(neg_prob))])
                scores = np.concatenate([pos_prob, neg_prob])
                rows.append({
                    "from_year": bucket_year, "to_year": next_year,
                    "auc": float(binary_auc(labels, scores)),
                    "n_positive": len(positive_ids), "n_negative": len(negative_ids),
                })

    return pd.DataFrame(rows)

gold_recurrence = recurrence_auc_per_transition(gold_model, gold_graph, gold_time_mean, gold_time_known, gold_final_entity_state)
predicted_recurrence = recurrence_auc_per_transition(predicted_model, predicted_graph, predicted_time_mean, predicted_time_known, predicted_final_entity_state)

print("gold_events -- entity-state drift by year:")
display(gold_drift)
print("gold_events -- next-year recurrence AUC by transition:")
display(gold_recurrence)

print("\npredicted_events -- entity-state drift by year:")
display(predicted_drift)
print("predicted_events -- next-year recurrence AUC by transition:")
display(predicted_recurrence)

if len(gold_drift) or len(predicted_drift):
    fig, ax = plt.subplots(figsize=(7, 4))
    if len(gold_drift):
        ax.plot(gold_drift["year"], gold_drift["drift"], marker="o", label="gold_events")
    if len(predicted_drift):
        ax.plot(predicted_drift["year"], predicted_drift["drift"], marker="o", label="predicted_events")
    ax.set_xlabel("year")
    ax.set_ylabel("mean entity-state drift")
    ax.set_title("How much entity representations moved from the previous year")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 10c. Watching an entity's embedding actually move

The drift chart above answers "how much did entity_state change each year." It doesn't show *which direction* — two years could have the same drift magnitude while one moves back toward where it started (oscillation, i.e. not much real change) and the other keeps moving further away (a genuinely drifting pattern). This cell captures the actual `entity_state` snapshot at the end of every chronological bucket (not just its movement) and projects a chosen entity's sequence of positions onto 2 dimensions with PCA, so you can see the path it traces year over year rather than just its speed.

In [ ]:
def entity_state_snapshots(model, graph, time_mean, time_known):
    """Re-run the chronological rollout in eval mode and keep the entity_state
    at the end of every dated year-bucket (not just its drift magnitude)."""
    buckets = graph["train_by_year"]
    entity_state = model.entity_embeddings.weight
    snapshots = {}

    model.eval()
    with torch.inference_mode():
        for bucket_year, bucket_rows in buckets:
            b_ids, b_years, b_known, b_prov = to_tensors(bucket_rows)
            out = model(b_ids, b_years, b_known, b_prov, time_mean, time_known,
                         entity_state_init=entity_state)
            entity_state = out["entity_state"]
            if bucket_year is not None:
                snapshots[bucket_year] = entity_state.detach().cpu().clone()

    return snapshots

def plot_entity_trajectory(role, value, snapshots, vocab, offsets, title_prefix=""):
    local_id = vocab[role].get(normalize(value) or EMPTY, 0)
    global_id = offsets[role] + local_id

    years_sorted = sorted(snapshots)
    vectors = np.stack([snapshots[year][global_id].numpy() for year in years_sorted])

    if len(years_sorted) < 2:
        print(f"'{value}' only has state recorded for {len(years_sorted)} year(s) -- "
              f"nothing to trace a path through yet.")
        return None

    # PCA via SVD (no sklearn dependency): project onto the top 2 directions of
    # variance across this entity's own trajectory.
    centered = vectors - vectors.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    components = vt[:min(2, vt.shape[0])]
    coords = centered @ components.T
    if coords.shape[1] == 1:
        coords = np.hstack([coords, np.zeros_like(coords)])

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(coords[:, 0], coords[:, 1], marker="o", linestyle="-", alpha=0.7)
    for i, year in enumerate(years_sorted):
        ax.annotate(str(year), (coords[i, 0], coords[i, 1]),
                     textcoords="offset points", xytext=(6, 4))
    ax.scatter(coords[0, 0], coords[0, 1], color="green", zorder=5, label="earliest")
    ax.scatter(coords[-1, 0], coords[-1, 1], color="red", zorder=5, label="latest")
    ax.set_title(f"{title_prefix}embedding trajectory: {role} = '{value}'")
    ax.set_xlabel("PC1 (of this entity's own trajectory)")
    ax.set_ylabel("PC2")
    ax.legend()
    plt.tight_layout()
    plt.show()

    step_distances = np.linalg.norm(np.diff(vectors, axis=0), axis=1)
    return pd.DataFrame({
        "from_year": years_sorted[:-1], "to_year": years_sorted[1:],
        "step_distance": step_distances,
    })

gold_snapshots = entity_state_snapshots(gold_model, gold_graph, gold_time_mean, gold_time_known)
predicted_snapshots = entity_state_snapshots(predicted_model, predicted_graph, predicted_time_mean, predicted_time_known)

# Pick whichever ai_system value shows up in the most distinct years, so the
# trajectory has something to show. Swap this for any (role, value) you care about.
def most_recurring_value(role, hyperedges):
    counts = {}
    for edge in hyperedges:
        if edge["year"] <= 0:
            continue
        value = edge["values"][role]
        counts.setdefault(value, set()).add(edge["year"])
    if not counts:
        return None
    return max(counts, key=lambda v: len(counts[v]))

example_role = "ai_system"
example_value = most_recurring_value(example_role, gold_hyperedges)

if example_value:
    print(f"Tracing '{example_value}' ({example_role}) across years it appears in:")
    plot_entity_trajectory(example_role, example_value, gold_snapshots, gold_graph["vocab"], gold_graph["offsets"], "gold_events -- ")
else:
    print("No dated entity recurs across years in gold_hyperedges -- nothing to trace yet.")

## 11. Learned embeddings and analogical retrieval

In [ ]:
def event_embeddings(model, rows, time_mean, time_known):
    if not rows:
        return pd.DataFrame()

    global_ids, years, known, provenance = to_tensors(rows)

    model.eval()

    with torch.inference_mode():
        vectors = model(
            global_ids,
            years,
            known,
            provenance,
            time_mean,
            time_known,
        )["embedding"].cpu().numpy()

    return pd.DataFrame({
        "event_id": [row["event_id"] for row in rows],
        "year": [row["raw_year"] for row in rows],
        "embedding": [vector.tolist() for vector in vectors],
    })

def retrieve_similar(event_id, frame, top_k=5):
    ids = frame["event_id"].tolist()

    if event_id not in ids:
        raise KeyError(event_id)

    matrix = np.vstack(frame["embedding"].apply(np.asarray))
    matrix = matrix / np.clip(
        np.linalg.norm(matrix, axis=1, keepdims=True),
        1e-12,
        None,
    )

    query_index = ids.index(event_id)
    scores = matrix @ matrix[query_index]
    result = []

    for index in np.argsort(-scores):
        if ids[index] == event_id:
            continue

        result.append({
            "event_id": ids[index],
            "similarity": float(scores[index]),
            "gold_harm_category": gold_by_id[ids[index]]["harm_category"],
            "gold_ai_system": gold_by_id[ids[index]]["ai_system"],
            "year": int(frame.iloc[index]["year"]),
        })

        if len(result) == top_k:
            break

    return pd.DataFrame(result)

gold_embedding_df = event_embeddings(
    gold_model,
    gold_graph["train"] + gold_graph["validation"] + gold_graph["test"],
    gold_time_mean,
    gold_time_known,
)

predicted_embedding_df = event_embeddings(
    predicted_model,
    predicted_graph["train"] + predicted_graph["validation"] + predicted_graph["test"],
    predicted_time_mean,
    predicted_time_known,
)

gold_embedding_df.to_json(
    OUTPUT_DIR / "gold_event_embeddings.jsonl",
    orient="records",
    lines=True,
)

predicted_embedding_df.to_json(
    OUTPUT_DIR / "predicted_event_embeddings.jsonl",
    orient="records",
    lines=True,
)

query_id = gold_embedding_df.iloc[0]["event_id"]
print("Query:", query_id)
display(retrieve_similar(query_id, gold_embedding_df, top_k=5))

## 12. Save models and run manifest

In [ ]:
torch.save(
    {
        "model_state_dict": gold_model.state_dict(),
        "roles": ROLES,
        "vocab": gold_graph["vocab"],
        "offsets": gold_graph["offsets"],
        "year_min": gold_graph["year_min"],
        "year_max": gold_graph["year_max"],
        # Entity state as of the end of chronological training -- inference should
        # start from this (and keep evolving it) rather than the raw embedding
        # parameter, so it reflects everything the model has "seen happen" in order.
        "final_entity_state": gold_final_entity_state,
        "phtkg_config": best_phtkg_config,
    },
    OUTPUT_DIR / "gold_phtkg.pt",
)

torch.save(
    {
        "model_state_dict": predicted_model.state_dict(),
        "roles": ROLES,
        "vocab": predicted_graph["vocab"],
        "offsets": predicted_graph["offsets"],
        "year_min": predicted_graph["year_min"],
        "year_max": predicted_graph["year_max"],
        "final_entity_state": predicted_final_entity_state,
        "phtkg_config": best_phtkg_config,
    },
    OUTPUT_DIR / "predicted_phtkg.pt",
)

gold_drift.to_csv(OUTPUT_DIR / "gold_entity_drift_by_year.csv", index=False)
predicted_drift.to_csv(OUTPUT_DIR / "predicted_entity_drift_by_year.csv", index=False)
gold_recurrence.to_csv(OUTPUT_DIR / "gold_recurrence_auc_by_transition.csv", index=False)
predicted_recurrence.to_csv(OUTPUT_DIR / "predicted_recurrence_auc_by_transition.csv", index=False)

manifest = {
    "input_records": len(reports),
    "gold_records": len(gold_events_eval),
    "qwen_enabled": RUN_QWEN,
    "inference_protocol": "single-pass-one-qwen-call-per-report",
    "classification_policy": "copy input classification/classifications verbatim; no prediction or verification",
    "nli_enabled": False,
    "trainable_module": "Auto-PHTKG" if AUTO_TUNE else "PHTKG",
    "auto_tune_enabled": AUTO_TUNE,
    "auto_tune_trials": AUTO_TUNE_TRIALS if AUTO_TUNE else 0,
    "selected_phtkg_config": best_phtkg_config,
    "extraction_evaluation": "field exact match, token F1, date score, and evidence containment",
    "gold_graph_experiment": "isolates graph-model performance",
    "predicted_graph_experiment": "measures end-to-end error propagation",
    "phtkg_objective": "observed-versus-corrupted temporal hyperedge prediction, "
                        "trained chronologically by year-bucket with a next-year "
                        "recurrence-forecast auxiliary objective (weight="
                        f"{best_phtkg_config.get('EXTRAPOLATION_WEIGHT', EXTRAPOLATION_WEIGHT)})",
    "pattern_learning": "explicit chronological entity-state evolution (drift logged "
                         "per year) plus a learned next-year recurrence predictor, "
                         "in addition to the implicit relational-temporal structure "
                         "in event/entity embeddings",
}

(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Saved outputs to:", OUTPUT_DIR)


## Component Wise Verification of PHTKG

In [ ]:
import matplotlib.pyplot as plt

def forward_with_trace(self, global_ids, years, known, provenance, time_mean, time_known):
    event_count = global_ids.shape[0]
    entity_state = self.entity_embeddings.weight
    event_state = self.event_seed.unsqueeze(0).expand(event_count, -1)

    years_column = years.unsqueeze(-1)
    known_column = known.unsqueeze(-1)
    observed_time = (
        years_column * self.time_linear_weight
        + self.time_linear_bias
        + torch.sin(years_column * self.time_periodic_weight + self.time_periodic_bias)
    )
    time_message = known_column * observed_time + (1 - known_column) * self.missing_time
    provenance_message = self.provenance_message(provenance.unsqueeze(-1))
    provenance_gate = torch.sigmoid(provenance).unsqueeze(-1)

    trace = {
        "attention": [], "event_states": [event_state.detach()],
        "temporal_weight": [], "message_norm": [],
        "time_message_norm": time_message.norm(dim=-1).detach(),
        "provenance_message_norm": provenance_message.norm(dim=-1).detach(),
    }
    for _ in range(self.layers):
        incident = entity_state[global_ids] + self.role_embeddings.unsqueeze(0)
        keys = self.entity_key(incident)
        values = self.entity_value(incident)
        query = self.event_query(event_state).unsqueeze(1)
        logits = (query * keys).sum(-1) / math.sqrt(self.dimension) + self.role_bias.unsqueeze(0)
        attention = F.softmax(logits, dim=1)
        trace["attention"].append(attention.detach())

        entity_message = (attention.unsqueeze(-1) * values).sum(1)
        event_state = self.event_update(entity_message + time_message + provenance_message, event_state)
        trace["event_states"].append(event_state.detach())

        aggregate = torch.zeros_like(entity_state)
        denominator = torch.zeros(entity_state.shape[0], 1, device=entity_state.device)
        layer_weights, layer_norms = [], []
        for role_index in range(len(ROLES)):
            ids = global_ids[:, role_index]
            temporal_known = known * time_known[ids]
            gap = torch.abs(years - time_mean[ids])
            decay = F.softplus(self.temporal_decay[role_index])
            temporal_weight = temporal_known * torch.exp(-decay * gap) + (1 - temporal_known)
            weight = temporal_weight * provenance_gate.squeeze(-1)
            message = self.event_to_entity[role_index](event_state) * weight.unsqueeze(-1)
            aggregate.index_add_(0, ids, message)
            denominator.index_add_(0, ids, weight.unsqueeze(-1))
            layer_weights.append(weight.detach())
            layer_norms.append(message.norm(dim=-1).detach())
        trace["temporal_weight"].append(layer_weights)
        trace["message_norm"].append(layer_norms)
        entity_input = aggregate / torch.clamp(denominator, min=1.0)
        proposed_state = self.entity_update(entity_input, entity_state)
        active_mask = denominator > 0
        entity_state = torch.where(active_mask, proposed_state, entity_state)

    trace["final_entity_state"] = entity_state.detach()
    trace["embedding"] = F.normalize(event_state, dim=-1).detach()
    trace["score"] = self.scorer(trace["embedding"]).squeeze(-1).detach()
    return trace

PHTKG.forward_with_trace = forward_with_trace

def encode_synthetic(graph, values, year, provenance=1.0):
    """Encode one synthetic event through an existing graph vocabulary (unseen -> __UNK__)."""
    ids = []
    for role in ROLES:
        local = graph["vocab"][role].get(normalize(values.get(role, "")) or EMPTY, 0)
        ids.append(graph["offsets"][role] + local)
    span = max(1, graph["year_max"] - graph["year_min"])
    known = float(year > 0)
    y = (year - graph["year_min"]) / span if year > 0 else 0.0
    return (
        torch.tensor([ids], dtype=torch.long, device=DEVICE),
        torch.tensor([y], dtype=torch.float32, device=DEVICE),
        torch.tensor([known], dtype=torch.float32, device=DEVICE),
        torch.tensor([float(provenance)], dtype=torch.float32, device=DEVICE),
    )

def fit_model(model, graph, label, epochs=60):
    """Generic trainer for later ablations, with the same leakage-safe corruption guard."""
    global CURRENT_REAL_SIGNATURES, CURRENT_CORRUPTION_WEIGHTS
    rows = graph["train"]
    cfg = getattr(model, "config", current_phtkg_config())
    CURRENT_CORRUPTION_WEIGHTS = dict(cfg.get("CORRUPTION_WEIGHTS", best_phtkg_config.get("CORRUPTION_WEIGHTS", CORRUPTION_WEIGHTS)))
    CURRENT_REAL_SIGNATURES = build_real_signatures(graph, splits=("train",))
    g_ids, g_years, g_known, g_prov, t_mean, t_known = graph_context(rows, graph["total_nodes"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg.get("LEARNING_RATE", LEARNING_RATE)), weight_decay=float(cfg.get("WEIGHT_DECAY", WEIGHT_DECAY)))
    for epoch in range(1, epochs + 1):
        model.train()
        neg_count = int(cfg.get("NEGATIVES", NEGATIVES))
        neg_ids = corrupt(g_ids, graph["vocab"], graph["offsets"], neg_count)
        pos = model(g_ids, g_years, g_known, g_prov, t_mean, t_known)["score"]
        neg = model(
            neg_ids,
            g_years.repeat_interleave(neg_count), g_known.repeat_interleave(neg_count),
            g_prov.repeat_interleave(neg_count), t_mean, t_known,
        )["score"]
        loss = (
            F.binary_cross_entropy_with_logits(pos, torch.ones_like(pos))
            + F.binary_cross_entropy_with_logits(neg, torch.zeros_like(neg))
        ) / 2
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.get("GRAD_CLIP", GRAD_CLIP)))
        optimizer.step()
    CURRENT_REAL_SIGNATURES = build_real_signatures(
        graph, splits=("train", "validation", "test")
    )
    result = evaluate(model, graph["test"] or graph["train"], t_mean, t_known, graph)
    print(f"{label} | test-AUC={result['auc']:.4f}")
    return result

verification_results = []


## HyperGraph vs Pair Wise KG

In [ ]:
class PairwiseBaseline(nn.Module):
    """DistMult scorer over all role-pair triples of an event (no joint hyperedge)."""
    def __init__(self, total_nodes, n_roles):
        super().__init__()
        d = EMBEDDING_DIM
        self.entity = nn.Embedding(total_nodes, d)
        self.relation = nn.Embedding(n_roles * n_roles, d)
        self.n_roles = n_roles
    def forward(self, global_ids):
        h = self.entity(global_ids)
        B, R, d = h.shape
        hi = h.unsqueeze(2).expand(B, R, R, d)
        hj = h.unsqueeze(1).expand(B, R, R, d)
        rel_idx = torch.arange(R * R, device=h.device).view(R, R)
        r = self.relation(rel_idx).unsqueeze(0).expand(B, R, R, d)
        return (hi * r * hj).sum(-1).mean(dim=(1, 2))

CURRENT_REAL_SIGNATURES = build_real_signatures(gold_graph, splits=("train",))  # leakage-safe training guard
pair_model = PairwiseBaseline(gold_graph["total_nodes"], len(ROLES)).to(DEVICE)
pair_opt = torch.optim.AdamW(pair_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
g_ids, g_years, g_known, g_prov = to_tensors(gold_graph["train"])
for epoch in range(1, EPOCHS + 1):
    pair_model.train()
    neg_ids = corrupt(g_ids, gold_graph["vocab"], gold_graph["offsets"], NEGATIVES)
    pos = pair_model(g_ids)
    neg = pair_model(neg_ids)
    loss = (
        F.binary_cross_entropy_with_logits(pos, torch.ones_like(pos))
        + F.binary_cross_entropy_with_logits(neg, torch.zeros_like(neg))
    ) / 2
    pair_opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(pair_model.parameters(), 1.5)
    pair_opt.step()

test_ids, _, _, _ = to_tensors(gold_graph["test"] or gold_graph["train"])
CURRENT_REAL_SIGNATURES = build_real_signatures(
    gold_graph, splits=("train", "validation", "test")
)  # evaluation guard may protect known test positives
neg_test = corrupt(test_ids, gold_graph["vocab"], gold_graph["offsets"], 3)
pair_model.eval()
with torch.inference_mode():
    pos_p = torch.sigmoid(pair_model(test_ids)).cpu().numpy()
    neg_p = torch.sigmoid(pair_model(neg_test)).cpu().numpy()
pair_auc = binary_auc(
    np.concatenate([np.ones(len(pos_p)), np.zeros(len(neg_p))]),
    np.concatenate([pos_p, neg_p]),
)
hyper_auc = float(comparison_df.query("experiment=='gold_events' and split=='test'")["auc"].iloc[0])
print(f"Pairwise KG test-AUC : {pair_auc:.4f}")
print(f"Hypergraph  test-AUC : {hyper_auc:.4f}")
verification_results.append({
    "Component": "Hypergraph",
    "Intended behavior": "Preserve the complete incident jointly",
    "Experiment": "Pairwise DistMult baseline vs. hypergraph, same corruption protocol",
    "Observed": f"pairwise AUC={pair_auc:.3f}, hypergraph AUC={hyper_auc:.3f}",
    "Verdict": "✅" if hyper_auc >= pair_auc else "⚠️ inspect on full data",
})

In [ ]:
def build_graph_from_split(train_edges, validation_edges, test_edges):
    vocab, offsets, current = {}, {}, 0
    for role in ROLES:
        values = sorted({normalize(e["values"][role]) or EMPTY for e in train_edges})
        role_vocab = {"__UNK__": 0, EMPTY: 1}
        for v in values:
            if v not in role_vocab:
                role_vocab[v] = len(role_vocab)
        vocab[role] = role_vocab
        offsets[role] = current
        current += len(role_vocab)

    years = [e["year"] for e in train_edges if e["year"] > 0]
    year_min = min(years) if years else 2000
    year_max = max(years) if years else year_min + 1

    def encode(items):
        rows = []
        for e in items:
            ids = [offsets[r] + vocab[r].get(normalize(e["values"][r]) or EMPTY, 0) for r in ROLES]
            known = float(e["year"] > 0)
            y = (e["year"] - year_min) / max(1, year_max - year_min) if e["year"] > 0 else 0.0
            rows.append({"event_id": e["event_id"], "global_ids": ids, "year": float(y),
                         "known": known, "raw_year": e["year"], "provenance": float(e["provenance"])})
        return rows

    train_rows = encode(train_edges)
    buckets = {}
    for row in train_rows:
        key = row["raw_year"] if row["raw_year"] > 0 else None
        buckets.setdefault(key, []).append(row)
    ordered = sorted(k for k in buckets if k is not None)
    train_by_year = ([(None, buckets[None])] if None in buckets else []) + [(k, buckets[k]) for k in ordered]

    return {"train": train_rows, "train_by_year": train_by_year,
            "validation": encode(validation_edges), "test": encode(test_edges),
            "vocab": vocab, "offsets": offsets, "total_nodes": current,
            "year_min": year_min, "year_max": year_max}

def chronological_folds(edges, n_folds=4, min_history_years=3):
    """
    Strict year-grouped expanding-window CV.

    - Events with unknown year are excluded from temporal CV.
    - A calendar year is never split across train/validation/test.
    - For each fold:
          all earlier years except the most recent -> train
          most recent earlier year              -> validation
          next held-out year                    -> test
    - The last `n_folds` eligible years are used as test years.

    This guarantees:
        max(train_year) < min(validation_year) < min(test_year)
    """
    dated = [e for e in edges if e["year"] > 0]
    undated_count = sum(e["year"] <= 0 for e in edges)

    by_year = {}
    for e in dated:
        by_year.setdefault(int(e["year"]), []).append(e)

    years = sorted(by_year)

    # Need at least min_history_years earlier years before a year can be tested.
    eligible_test_years = years[min_history_years:]
    test_years = eligible_test_years[-n_folds:]

    folds = []
    for test_year in test_years:
        test_idx = years.index(test_year)
        prior_years = years[:test_idx]

        # Reserve the immediately preceding complete year for validation.
        val_year = prior_years[-1]
        train_years = prior_years[:-1]

        train_edges = [e for y in train_years for e in by_year[y]]
        validation_edges = list(by_year[val_year])
        test_edges = list(by_year[test_year])

        if not train_edges or not validation_edges or not test_edges:
            continue

        # Leakage guards: complete years must remain strictly ordered.
        assert max(e["year"] for e in train_edges) < min(e["year"] for e in validation_edges)
        assert max(e["year"] for e in validation_edges) < min(e["year"] for e in test_edges)
        assert all(e["year"] > 0 for e in train_edges + validation_edges + test_edges)

        folds.append({
            "train": train_edges,
            "validation": validation_edges,
            "test": test_edges,
            "train_years": train_years,
            "validation_years": [val_year],
            "test_years": [test_year],
        })

    return folds, undated_count

def run_chronological_cv(edges, label, n_folds=4):
    global CURRENT_REAL_SIGNATURES

    folds, undated_count = chronological_folds(edges, n_folds=n_folds)

    print(
        f"{label}: strict year-grouped chronological CV "
        f"({len(folds)} folds; {undated_count} undated event(s) excluded)"
    )

    results = []
    for i, fold in enumerate(folds, 1):
        train_e = fold["train"]
        val_e = fold["validation"]
        test_e = fold["test"]

        graph = build_graph_from_split(train_e, val_e, test_e)
        # train_phtkg applies a TRAIN-only corruption guard internally.
        _, _, metrics, *_ = train_phtkg(graph, f"{label}_fold{i}")
        auc = metrics.query("split == 'test'").iloc[0]["auc"]

        train_year_min = min(fold["train_years"])
        train_year_max = max(fold["train_years"])
        val_year = fold["validation_years"][0]
        test_year = fold["test_years"][0]

        results.append({
            "fold": i,
            "train_years": f"{train_year_min}-{train_year_max}",
            "validation_year": val_year,
            "test_year": test_year,
            "n_train": len(train_e),
            "n_validation": len(val_e),
            "n_test": len(test_e),
            "test_auc": auc,
        })

        print(
            f"fold {i}: train={train_year_min}-{train_year_max} "
            f"(n={len(train_e)}), val={val_year} (n={len(val_e)}), "
            f"test={test_year} (n={len(test_e)}), AUC={auc:.4f}"
        )

    df = pd.DataFrame(results)

    if len(df):
        std = df["test_auc"].std(ddof=1) if len(df) > 1 else 0.0
        print(
            f"\n{label}: mean AUC={df['test_auc'].mean():.4f} "
            f"± {std:.4f} over {len(df)} folds"
        )
        display(df)
    else:
        print(f"\n{label}: not enough dated years to construct chronological folds.")

    return df

gold_cv = run_chronological_cv(gold_hyperedges, "gold_events", n_folds=4)
predicted_cv = run_chronological_cv(predicted_hyperedges, "predicted_events", n_folds=4)

## Role Embeddings Verification
Role embeddings: same entity in different roles + role geometry

In [ ]:
shared, cos_cross_role = None, float("nan")
for i, r1 in enumerate(ROLES):
    for r2 in ROLES[i + 1:]:
        common = (set(gold_graph["vocab"][r1]) & set(gold_graph["vocab"][r2])) - {"__UNK__", EMPTY}
        if common:
            shared = (sorted(common)[0], r1, r2)
            break
    if shared:
        break

table = gold_model.entity_embeddings.weight.detach()  # FIX: moved out of the `if shared` block
if shared:
    value, r1, r2 = shared
    id1 = gold_graph["offsets"][r1] + gold_graph["vocab"][r1][value]
    id2 = gold_graph["offsets"][r2] + gold_graph["vocab"][r2][value]
    cos_cross_role = F.cosine_similarity(table[id1], table[id2], dim=0).item()
    print(f'"{value}" as {r1} vs. as {r2}: cosine = {cos_cross_role:.4f}')
else:
    print("No surface value appears in two role vocabularies; role separation is enforced structurally "
          "(per-role vocabularies give every (value, role) pair its own node).")

role_cos = F.cosine_similarity(
    gold_model.role_embeddings.unsqueeze(1), gold_model.role_embeddings.unsqueeze(0), dim=-1
).detach().cpu().numpy()
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(role_cos, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(ROLES)), ROLES, rotation=45, ha="right")
ax.set_yticks(range(len(ROLES)), ROLES)
ax.set_title("Cosine similarity between learned role embeddings")
fig.colorbar(im)
plt.tight_layout(); plt.show()

from sklearn.decomposition import PCA
points, labels_role, labels_name = [], [], []
for role in ROLES:
    for value, local_id in gold_graph["vocab"][role].items():
        if value in ("__UNK__", EMPTY):
            continue
        points.append(table[gold_graph["offsets"][role] + local_id].cpu().numpy())
        labels_role.append(role); labels_name.append(value)
proj = PCA(n_components=2, random_state=SEED).fit_transform(np.vstack(points))
fig, ax = plt.subplots(figsize=(9, 6))
for role in ROLES:
    mask = [i for i, r in enumerate(labels_role) if r == role]
    ax.scatter(proj[mask, 0], proj[mask, 1], label=role, s=40, alpha=0.8)
ax.legend(fontsize=8); ax.set_title("Trained entity embeddings (PCA), colored by role")
plt.tight_layout(); plt.show()

verification_results.append({
    "Component": "Role embeddings",
    "Intended behavior": "Entity representation depends on its role",
    "Experiment": "Same surface value in two roles; role-similarity heatmap; PCA by role",
    "Observed": (f'cross-role cosine={cos_cross_role:.3f}' if shared else "structural separation (per-role vocab)"),
    "Verdict": "✅" if (not shared or abs(cos_cross_role) < 0.9) else "⚠️ roles collapsed",
})

## Attention Verification

In [ ]:

rows_all = gold_graph["train"] + gold_graph["validation"] + gold_graph["test"]
g_ids, g_years, g_known, g_prov = to_tensors(rows_all)
gold_model.eval()
with torch.inference_mode():
    trace = gold_model.forward_with_trace(g_ids, g_years, g_known, g_prov, gold_time_mean, gold_time_known)

attn_last = trace["attention"][-1].cpu().numpy()
mean_attn = attn_last.mean(axis=0)
attn_df = pd.DataFrame({"role": ROLES, "mean_attention": mean_attn}).sort_values(
    "mean_attention", ascending=False)
display(attn_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(attn_df["role"], attn_df["mean_attention"])
axes[0].set_title("Mean attention per role (final layer)")
axes[0].tick_params(axis="x", rotation=45)
im = axes[1].imshow(attn_last, aspect="auto", cmap="viridis")
axes[1].set_xticks(range(len(ROLES)), ROLES, rotation=45, ha="right")
axes[1].set_yticks(range(len(rows_all)), [r["event_id"] for r in rows_all], fontsize=7)
axes[1].set_title("Attention per event")
fig.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

top3 = set(attn_df["role"].head(3))
informative = {"ai_system", "action", "harm_category", "affected_group", "organization"}
verification_results.append({
    "Component": "Attention",
    "Intended behavior": "Focus on informative participants",
    "Experiment": "Mean attention per role + per-event heatmap",
    "Observed": "top-3 roles: " + ", ".join(attn_df["role"].head(3)),
    "Verdict": "✅" if len(top3 & informative) >= 2 and "source" not in top3 else "⚠️ inspect heatmap",
})

## Temporal decay kernel per role

Intended behavior is events temporally distant from an entity's centroid influence it less

In [ ]:

span = max(1, gold_graph["year_max"] - gold_graph["year_min"])
gaps_years = np.linspace(0, 6, 60)
fig, ax = plt.subplots(figsize=(8, 5))
kappa_table = {}
for role_index, role in enumerate(ROLES):
    decay = F.softplus(gold_model.temporal_decay[role_index]).item()
    kappa = np.exp(-decay * gaps_years / span)
    ax.plot(gaps_years, kappa, label=f"{role} (γ={decay:.3f})")
    kappa_table[role] = [float(np.exp(-decay * g / span)) for g in [0, 1, 3, 5]]
ax.set_xlabel("Temporal distance from entity centroid (years)")
ax.set_ylabel("Temporal weight κ")
ax.set_title("Learned temporal-decay kernels")
ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
display(pd.DataFrame(kappa_table, index=["gap 0y", "gap 1y", "gap 3y", "gap 5y"]).round(3))

monotone = all(
    kappa_table[r] == sorted(kappa_table[r], reverse=True) for r in ROLES
)
verification_results.append({
    "Component": "Temporal decay",
    "Intended behavior": "Distant events influence entities less",
    "Experiment": "Plot learned κ vs. temporal distance per role",
    "Observed": f"decay rates γ: " + ", ".join(
        f"{r}={F.softplus(gold_model.temporal_decay[i]).item():.2f}" for i, r in enumerate(ROLES)),
    "Verdict": "✅" if monotone else "⚠️",
})

## Temporal encoding Verification
Intended behavior is that changing only the year changes the event embedding smoothly so nearby years will have higher similarity & distant years will have lower similarity.


In [ ]:
template = next(e for e in gold_hyperedges if e["year"] > 0)
years_to_test = [2018, 2020, 2024]
embs = {}
gold_model.eval()
with torch.inference_mode():
    for y in years_to_test:
        s_ids, s_y, s_k, s_p = encode_synthetic(gold_graph, template["values"], y, provenance=1.0)
        embs[y] = gold_model(s_ids, s_y, s_k, s_p, gold_time_mean, gold_time_known)["embedding"][0]

pairs = [(2018, 2020), (2020, 2024), (2018, 2024)]
sims = {p: F.cosine_similarity(embs[p[0]], embs[p[1]], dim=0).item() for p in pairs}
for (a, b), s in sims.items():
    print(f"cos(E_{a}, E_{b}) = {s:.4f}")

smooth = sims[(2018, 2024)] <= max(sims[(2018, 2020)], sims[(2020, 2024)]) + 1e-6
verification_results.append({
    "Component": "Temporal encoding",
    "Intended behavior": "Time changes the embedding smoothly",
    "Experiment": "Identical event, years 2018/2020/2024; pairwise cosine",
    "Observed": ", ".join(f"{a}-{b}: {s:.3f}" for (a, b), s in sims.items()),
    "Verdict": "✅" if smooth else "⚠️ non-monotonic (check with more training data)",
})

## Provenance message Verification

In [ ]:
gold_model.eval()
results_prov = {}
with torch.inference_mode():
    for p in (0.95, 0.15):
        s_ids, s_y, s_k, s_p = encode_synthetic(
            gold_graph, template["values"], template["year"], provenance=p)
        tr = gold_model.forward_with_trace(s_ids, s_y, s_k, s_p, gold_time_mean, gold_time_known)
        results_prov[p] = {
            "prov_msg_norm": float(tr["provenance_message_norm"][0]),
            "score": float(torch.sigmoid(tr["score"][0])),
            "embedding": tr["embedding"][0],
        }
cos_shift = 1 - F.cosine_similarity(
    results_prov[0.95]["embedding"], results_prov[0.15]["embedding"], dim=0).item()
print(f"provenance-message norm  p=0.95: {results_prov[0.95]['prov_msg_norm']:.4f} | "
      f"p=0.15: {results_prov[0.15]['prov_msg_norm']:.4f}")
print(f"compatibility score      p=0.95: {results_prov[0.95]['score']:.4f} | "
      f"p=0.15: {results_prov[0.15]['score']:.4f}")
print(f"cosine distance between the two embeddings: {cos_shift:.4f}")
verification_results.append({
    "Component": "Provenance message",
    "Intended behavior": "Confidence enters the event semantics",
    "Experiment": "Same event, p=0.95 vs. p=0.15",
    "Observed": f"embedding distance={cos_shift:.3f}, "
                f"score {results_prov[0.15]['score']:.3f}→{results_prov[0.95]['score']:.3f}",
    "Verdict": "✅" if cos_shift > 1e-4 else "⚠️ provenance has no effect",
})

## Provenance gate — direct propagation verification

This diagnostic tests the **actual gate used by PHTKG propagation**, rather than correlating provenance with a downstream message norm that is also affected by the event GRU, learned projections, attention, and temporal context.

For a fixed set of events, it verifies that increasing provenance confidence monotonically increases the gated event-to-entity propagation weight

\[w_{e\to v}=\kappa(\Delta t)\,\sigma(p_e).\]

The learned provenance-message pathway is verified separately above; this cell isolates the gate itself.


In [ ]:
# Direct verification of the provenance gate used inside PHTKG.forward().
# This intentionally tests the gate BEFORE downstream GRU/projection effects.

from scipy.stats import pearsonr, spearmanr

probe_rows = gold_graph["test"] or gold_graph["validation"] or gold_graph["train"]
p_ids, p_years, p_known, p_prov = to_tensors(probe_rows)

provenance_ps = np.linspace(0.05, 0.95, 10)
prov_rows = []

gold_model.eval()
with torch.inference_mode():
    for p in provenance_ps:
        gate_value = float(torch.sigmoid(torch.tensor(float(p), device=DEVICE)).item())
        role_weight_means = []

        for role_index, role in enumerate(ROLES):
            ids = p_ids[:, role_index]
            temporal_known = p_known * gold_time_known[ids]
            gap = torch.abs(p_years - gold_time_mean[ids])
            decay = F.softplus(gold_model.temporal_decay[role_index])
            temporal_weight = (
                temporal_known * torch.exp(-decay * gap)
                + (1 - temporal_known)
            )
            gated_weight = temporal_weight * gate_value
            role_weight_means.append(float(gated_weight.mean().item()))

        prov_rows.append({
            "provenance": float(p),
            "sigmoid_gate": gate_value,
            "mean_gated_propagation_weight": float(np.mean(role_weight_means)),
            "min_role_weight": float(np.min(role_weight_means)),
            "max_role_weight": float(np.max(role_weight_means)),
        })

provenance_gate_df = pd.DataFrame(prov_rows)
pearson_gate = pearsonr(
    provenance_gate_df["sigmoid_gate"],
    provenance_gate_df["mean_gated_propagation_weight"],
)[0]
spearman_gate = spearmanr(
    provenance_gate_df["provenance"],
    provenance_gate_df["mean_gated_propagation_weight"],
).correlation
monotonic_gate = bool(
    np.all(np.diff(provenance_gate_df["mean_gated_propagation_weight"].to_numpy()) > 0)
)

print(f"Direct provenance-gate Pearson r : {pearson_gate:.6f}")
print(f"Direct provenance-gate Spearman ρ: {spearman_gate:.6f}")
print(f"Strictly monotonic propagation    : {monotonic_gate}")
print(
    "Gate range: "
    f"{provenance_gate_df['sigmoid_gate'].min():.4f} → "
    f"{provenance_gate_df['sigmoid_gate'].max():.4f}"
)
display(provenance_gate_df.round(6))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(
    provenance_gate_df["provenance"],
    provenance_gate_df["mean_gated_propagation_weight"],
    "o-",
)
ax.set_xlabel("Provenance confidence p")
ax.set_ylabel("Mean gated propagation weight")
ax.set_title("Direct provenance-conditioned propagation")
plt.tight_layout(); plt.show()

provenance_gate_df.to_csv(OUTPUT_DIR / "provenance_gate_direct_diagnostic.csv", index=False)

verification_results.append({
    "Component": "Provenance gate",
    "Intended behavior": "Higher-confidence events receive stronger propagation weights",
    "Experiment": "Direct sweep of p in the implemented weight κ(Δt)·sigmoid(p)",
    "Observed": (
        f"Pearson r={pearson_gate:.4f}, Spearman rho={spearman_gate:.4f}, "
        f"strictly monotonic={monotonic_gate}"
    ),
    "Verdict": "✅" if monotonic_gate and pearson_gate > 0.999 else "⚠️ inspect direct gate",
})


## Event GRU Verification

In [ ]:
with torch.inference_mode():
    trace = gold_model.forward_with_trace(g_ids, g_years, g_known, g_prov, gold_time_mean, gold_time_known)
states = trace["event_states"]   # [layer 0 (seed), layer 1, ..., layer L]
print("Event-state evolution (mean over events):")
stab = []
for l in range(1, len(states)):
    cos = F.cosine_similarity(states[l - 1], states[l], dim=-1).mean().item()
    stab.append(cos)
    print(f"  layer {l-1} → layer {l}: cosine = {cos:.4f}, "
          f"state norm = {states[l].norm(dim=-1).mean():.3f}")

class MLPCell(nn.Module):
    """Drop-in replacement for GRUCell: same (input, hidden) interface, no gating/memory."""
    def __init__(self, d):
        super().__init__()
        self.lin = nn.Linear(d, d)
    def forward(self, x, h):
        return torch.tanh(self.lin(x) + h)

torch.manual_seed(SEED)
mlp_model = PHTKG(gold_graph["total_nodes"], config=best_phtkg_config).to(DEVICE)
mlp_model.event_update = MLPCell(gold_model.dimension).to(DEVICE)
mlp_result = fit_model(mlp_model, gold_graph, "ablation: MLP event update")
gru_auc = float(comparison_df.query("experiment=='gold_events' and split=='test'")["auc"].iloc[0])
print(f"GRU test-AUC = {gru_auc:.4f} | MLP test-AUC = {mlp_result['auc']:.4f}")
verification_results.append({
    "Component": "Event GRU",
    "Intended behavior": "Accumulate information across layers",
    "Experiment": "Layer-to-layer cosine; GRU vs. MLP ablation",
    "Observed": f"layer cosines {[round(c,3) for c in stab]}, GRU AUC={gru_auc:.3f}, MLP AUC={mlp_result['auc']:.3f}",
    "Verdict": "✅" if gru_auc >= mlp_result["auc"] else "⚠️ recheck on full data",
})

##  Entity GRU / entity embeddings

In [ ]:
torch.manual_seed(SEED + 1)
untrained = PHTKG(gold_graph["total_nodes"]).to(DEVICE)

target_value, target_role = None, None
for cand in ["facial-recognition system", "ai chatbot"]:
    for role in ROLES:
        if cand in gold_graph["vocab"][role]:
            target_value, target_role = cand, role
            break
    if target_value:
        break

def knn(model, node_id, k=5):
    table = model.entity_embeddings.weight.detach()
    sims = F.cosine_similarity(table, table[node_id].unsqueeze(0), dim=-1)
    top = sims.topk(k + 1).indices[1:]
    inv = {}
    for role in ROLES:
        for value, local in gold_graph["vocab"][role].items():
            inv[gold_graph["offsets"][role] + local] = f"{value} ({role})"
    return [inv.get(int(i), str(int(i))) for i in top]

if target_value:
    node_id = gold_graph["offsets"][target_role] + gold_graph["vocab"][target_role][target_value]
    print(f'Target entity: "{target_value}" as {target_role}')
    print("Before training:", knn(untrained, node_id))
    print("After training :", knn(gold_model, node_id))
    verification_results.append({
        "Component": "Entity GRU / entity embeddings",
        "Intended behavior": "Entities absorb their incident context",
        "Experiment": f'Nearest neighbors of "{target_value}" before vs. after training',
        "Observed": "after: " + "; ".join(knn(gold_model, node_id)[:3]),
        "Verdict": "🔍 inspect — neighbors should be co-participants, not random",
    })
else:
    print("Target entity not in vocabulary; pick any value from gold_graph['vocab'].")

## Compatibility scorer — aggregate role-wise corruption sensitivity

A single event can be unrepresentative, and sigmoid scores may saturate near 0 or 1 even when their **raw logits remain well separated**. This diagnostic therefore evaluates the full held-out gold test set and corrupts each role independently with same-role alternatives.

The primary statistic is a **paired AUC / win rate**: how often the real hyperedge receives a higher raw compatibility logit than its own corrupted counterpart. Raw-logit margin is reported alongside probability drop.


In [ ]:
# Aggregate, paired corruption-sensitivity diagnostic over the full held-out test set.
# We use same-role substitutions so each negative remains type-compatible.
# Primary metric = paired real-vs-own-corruption win rate on RAW LOGITS,
# which remains informative even when sigmoid probabilities are saturated.

def aggregate_role_corruption_sensitivity(
    model, graph, rows, time_mean, time_known, entity_state_init=None,
    draws_per_role=5, seed=SEED + 2026,
):
    if not rows:
        return pd.DataFrame(), pd.DataFrame(), {}

    rng = random.Random(seed)
    real_signatures = build_real_signatures(
        graph, splits=("train", "validation", "test")
    )
    details = []

    model.eval()
    state = entity_state_init

    with torch.inference_mode():
        for row in rows:
            r_ids, r_y, r_k, r_p = to_tensors([row])
            real_logit = float(model(
                r_ids, r_y, r_k, r_p, time_mean, time_known,
                entity_state_init=state,
            )["score"][0].item())
            real_prob = float(torch.sigmoid(torch.tensor(real_logit)).item())

            for role_index, role in enumerate(ROLES):
                original_local = int(r_ids[0, role_index]) - graph["offsets"][role]
                valid_candidates = []

                for local in range(2, len(graph["vocab"][role])):
                    if local == original_local:
                        continue
                    candidate_ids = r_ids.clone()
                    candidate_ids[0, role_index] = graph["offsets"][role] + local
                    signature = tuple(int(x) for x in candidate_ids[0].tolist())
                    if signature not in real_signatures:
                        valid_candidates.append(local)

                if not valid_candidates:
                    continue

                if len(valid_candidates) >= draws_per_role:
                    chosen = rng.sample(valid_candidates, draws_per_role)
                else:
                    chosen = [rng.choice(valid_candidates) for _ in range(draws_per_role)]

                for local in chosen:
                    corrupted = r_ids.clone()
                    corrupted[0, role_index] = graph["offsets"][role] + local
                    corrupt_logit = float(model(
                        corrupted, r_y, r_k, r_p, time_mean, time_known,
                        entity_state_init=state,
                    )["score"][0].item())
                    corrupt_prob = float(torch.sigmoid(torch.tensor(corrupt_logit)).item())
                    margin = real_logit - corrupt_logit
                    win = 1.0 if margin > 0 else 0.5 if margin == 0 else 0.0

                    details.append({
                        "event_id": row["event_id"],
                        "role": role,
                        "real_logit": real_logit,
                        "corrupted_logit": corrupt_logit,
                        "logit_margin": margin,
                        "real_probability": real_prob,
                        "corrupted_probability": corrupt_prob,
                        "probability_drop": real_prob - corrupt_prob,
                        "paired_win": win,
                    })

    detail_df = pd.DataFrame(details)
    if detail_df.empty:
        return detail_df, pd.DataFrame(), {}

    role_summary = (
        detail_df.groupby("role", sort=False)
        .agg(
            comparisons=("paired_win", "size"),
            paired_auc=("paired_win", "mean"),
            mean_logit_margin=("logit_margin", "mean"),
            median_logit_margin=("logit_margin", "median"),
            positive_margin_rate=("logit_margin", lambda x: float((x > 0).mean())),
            mean_probability_drop=("probability_drop", "mean"),
        )
        .reset_index()
        .sort_values("paired_auc", ascending=False)
    )

    overall = {
        "comparisons": int(len(detail_df)),
        "paired_auc": float(detail_df["paired_win"].mean()),
        "mean_logit_margin": float(detail_df["logit_margin"].mean()),
        "median_logit_margin": float(detail_df["logit_margin"].median()),
        "positive_margin_rate": float((detail_df["logit_margin"] > 0).mean()),
        "mean_probability_drop": float(detail_df["probability_drop"].mean()),
    }
    return detail_df, role_summary, overall

CURRENT_REAL_SIGNATURES = build_real_signatures(
    gold_graph, splits=("train", "validation", "test")
)
aggregate_scorer_details, aggregate_scorer_by_role, aggregate_scorer_overall = (
    aggregate_role_corruption_sensitivity(
        gold_model,
        gold_graph,
        gold_graph["test"] or gold_graph["validation"] or gold_graph["train"],
        gold_time_mean,
        gold_time_known,
        entity_state_init=gold_final_entity_state.to(DEVICE),
        draws_per_role=5,
    )
)

print("Aggregate compatibility/corruption diagnostic")
print(f"Comparisons            : {aggregate_scorer_overall.get('comparisons', 0)}")
print(f"Paired AUC / win rate  : {aggregate_scorer_overall.get('paired_auc', float('nan')):.4f}")
print(f"Mean raw-logit margin  : {aggregate_scorer_overall.get('mean_logit_margin', float('nan')):.4f}")
print(f"Median raw-logit margin: {aggregate_scorer_overall.get('median_logit_margin', float('nan')):.4f}")
print(f"Positive-margin rate   : {aggregate_scorer_overall.get('positive_margin_rate', float('nan')):.4f}")
print(f"Mean sigmoid drop      : {aggregate_scorer_overall.get('mean_probability_drop', float('nan')):.6f}")
display(aggregate_scorer_by_role.round(4))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
plot_df = aggregate_scorer_by_role.sort_values("paired_auc")
ax.barh(plot_df["role"], plot_df["paired_auc"])
ax.axvline(0.5, linestyle="--", linewidth=1)
ax.set_xlim(0, 1)
ax.set_xlabel("Paired AUC: P(real logit > own corruption)")
ax.set_title("Role-wise held-out corruption sensitivity")
plt.tight_layout(); plt.show()

aggregate_scorer_details.to_csv(
    OUTPUT_DIR / "compatibility_corruption_detailed.csv", index=False
)
aggregate_scorer_by_role.to_csv(
    OUTPUT_DIR / "compatibility_corruption_by_role.csv", index=False
)

overall_auc = aggregate_scorer_overall.get("paired_auc", float("nan"))
overall_margin = aggregate_scorer_overall.get("mean_logit_margin", float("nan"))
verification_results.append({
    "Component": "Compatibility scorer",
    "Intended behavior": "Real hyperedges outrank their own type-compatible corruptions",
    "Experiment": "Full held-out test set; 5 same-role corruptions per event/role; paired raw-logit comparison",
    "Observed": (
        f"paired AUC={overall_auc:.3f}, mean logit margin={overall_margin:.3f}, "
        f"positive-margin rate={aggregate_scorer_overall.get('positive_margin_rate', float('nan')):.3f}"
    ),
    "Verdict": "✅" if (overall_auc > 0.5 and overall_margin > 0) else "⚠️ inspect role-wise margins",
})


## Multi Hop Message Passing

In [ ]:
from collections import deque, defaultdict

adj = defaultdict(set)
for row in gold_graph["train"]:
    ids = row["global_ids"]
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            adj[ids[i]].add(ids[j]); adj[ids[j]].add(ids[i])
nodes = sorted(adj)

def bfs_dist(src):
    dist = {src: 0}
    q = deque([src])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

with torch.inference_mode():
    trace = gold_model.forward_with_trace(g_ids, g_years, g_known, g_prov, gold_time_mean, gold_time_known)
prop_state = trace["final_entity_state"]

buckets = defaultdict(list)
for a in nodes:
    da = bfs_dist(a)
    for b in nodes:
        if b <= a:
            continue
        d = da.get(b, 99)
        cos = F.cosine_similarity(prop_state[a], prop_state[b], dim=0).item()
        buckets[min(d, 3)].append(cos)

hop_df = pd.DataFrame([
    {"graph distance": {1: "1 (same event)", 2: "2 (one intermediate event)", 3: "≥3 / none"}[d],
     "pairs": len(v), "mean cosine": round(float(np.mean(v)), 4)}
    for d, v in sorted(buckets.items()) if v
])
display(hop_df)
ok = len(hop_df) >= 2 and hop_df["mean cosine"].iloc[0] >= hop_df["mean cosine"].iloc[-1]
verification_results.append({
    "Component": "Message passing",
    "Intended behavior": "Multi-hop relational context",
    "Experiment": "Entity-pair cosine vs. BFS distance in the event graph",
    "Observed": "; ".join(f"{r['graph distance']}: {r['mean cosine']:.3f}" for r in hop_df.to_dict('records')),
    "Verdict": "✅" if ok else "⚠️ no distance gradient (expected with 20 events)",
})

In [ ]:
verification_df = pd.DataFrame(verification_results)
display(verification_df)
verification_df.to_csv(OUTPUT_DIR / "component_verification.csv", index=False)

In [ ]:

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

FINAL_OUTPUT_PATH = OUTPUT_DIR / "predicted_events_with_learned_representation.jsonl"

TOP_K_SIMILAR = 5

# How n_patterns is chosen (see section 3 below):
#   "silhouette" : pick k maximizing silhouette score
#   "elbow"      : pick k at the elbow of the inertia curve
#   "combined"   : prefer k where silhouette and elbow agree;
#                  fall back to silhouette if they disagree
SELECTION_METHOD = "combined"

# Slope magnitude (events/year) above which a pattern is called
# "increasing" or "decreasing" rather than "stable".
FREQUENCY_TREND_THRESHOLD = 0.15

# ------------------------------------------------------------
# 0. RESOLVE THE ENTITY STATE TO EMBED WITH
#
# If chronological training produced predicted_final_entity_state,
# use it -- embeddings/temporal representation should reflect
# everything the model rolled through in order, not the raw
# embedding parameter. Falls back to None (the model's default)
# if that variable isn't present, e.g. an older non-chronological
# checkpoint.
# ------------------------------------------------------------

_entity_state_for_output = None
if "predicted_final_entity_state" in dir():
    _entity_state_for_output = predicted_final_entity_state.to(DEVICE)
    print("Using chronologically-rolled entity_state for embeddings/temporal representation.")
else:
    print("predicted_final_entity_state not found -- using the model's default "
          "(non-chronological) entity_embeddings.weight instead.")

# ------------------------------------------------------------
# 1. GET LEARNED EVENT EMBEDDINGS
# ------------------------------------------------------------

all_predicted_rows = (
    predicted_graph["train"]
    + predicted_graph["validation"]
    + predicted_graph["test"]
)

def learned_event_embeddings(model, rows, time_mean, time_known, entity_state_init=None):
    """Self-contained embedding computation (doesn't depend on an external
    event_embeddings() helper's signature), so it works whether or not
    that helper has been updated to accept entity_state_init."""
    if not rows:
        return pd.DataFrame()

    global_ids, years, known, provenance = to_tensors(rows)
    model.eval()

    with torch.inference_mode():
        vectors = model(
            global_ids, years, known, provenance, time_mean, time_known,
            entity_state_init=entity_state_init,
        )["embedding"].cpu().numpy()

    return pd.DataFrame({
        "event_id": [row["event_id"] for row in rows],
        "year": [row["raw_year"] for row in rows],
        "embedding": [vector.tolist() for vector in vectors],
    })

predicted_embeddings = learned_event_embeddings(
    predicted_model,
    all_predicted_rows,
    predicted_time_mean,
    predicted_time_known,
    entity_state_init=_entity_state_for_output,
)

if predicted_embeddings.empty:
    raise RuntimeError("No predicted event embeddings were produced.")

embedding_matrix = np.vstack(
    predicted_embeddings["embedding"].apply(np.asarray)
)

event_ids = predicted_embeddings["event_id"].tolist()

print("Learned event embeddings:", embedding_matrix.shape)

# ------------------------------------------------------------
# 2. NORMALIZE EMBEDDINGS
# ------------------------------------------------------------

embedding_matrix = embedding_matrix / np.clip(
    np.linalg.norm(embedding_matrix, axis=1, keepdims=True),
    1e-12,
    None,
)

# ------------------------------------------------------------
# 3. DISCOVER LATENT PATTERNS
#
# IMPORTANT:
# The PHTKG learns the representation.
# KMeans does NOT replace the neural model.
#
# It groups the learned representations into recurring
# latent relational/temporal structures.
#
# n_patterns is chosen data-drivenly, not from a fixed cap:
#   - "silhouette": pick k maximizing silhouette score
#   - "elbow":      pick k at the elbow of the inertia curve
#                    (max distance from the chord between the
#                    first and last inertia points)
#   - "combined":   compute both, prefer the k where they agree;
#                    if they disagree, fall back to silhouette
#                    (silhouette is the more principled metric
#                    when cluster shapes aren't spherical/even)
#
# Candidate K is searched up to n_events - 1 -- no arbitrary
# fixed ceiling.
# ------------------------------------------------------------

def _elbow_k(ks, inertias):
    """Return the k at max distance from the line joining the
    first and last (k, inertia) points -- the classic 'knee' pick."""
    if len(ks) < 3:
        return ks[0] if ks else None

    xs = np.array(ks, dtype=float)
    ys = np.array(inertias, dtype=float)

    # normalize so the geometry isn't dominated by inertia's scale
    xs_n = (xs - xs.min()) / max(xs.max() - xs.min(), 1e-12)
    ys_n = (ys - ys.min()) / max(ys.max() - ys.min(), 1e-12)

    p1 = np.array([xs_n[0], ys_n[0]])
    p2 = np.array([xs_n[-1], ys_n[-1]])
    line_vec = p2 - p1
    line_len = np.linalg.norm(line_vec)

    if line_len < 1e-12:
        return ks[0]

    line_unit = line_vec / line_len
    distances = []
    for x, y in zip(xs_n, ys_n):
        point_vec = np.array([x, y]) - p1
        proj_len = np.dot(point_vec, line_unit)
        proj_point = p1 + proj_len * line_unit
        distances.append(np.linalg.norm(np.array([x, y]) - proj_point))

    return ks[int(np.argmax(distances))]

n_events = len(embedding_matrix)

if n_events < 3:
    n_patterns = 1
    pattern_labels = np.zeros(n_events, dtype=int)
    cluster_centers = embedding_matrix.mean(axis=0, keepdims=True)
else:
    max_candidate_k = n_events - 1
    candidate_ks = [k for k in range(2, max_candidate_k + 1)]

    if not candidate_ks:
        n_patterns = 1
        pattern_labels = np.zeros(n_events, dtype=int)
        cluster_centers = embedding_matrix.mean(axis=0, keepdims=True)
    else:
        results = {}  # k -> {"labels", "centers", "inertia", "silhouette"}

        for k in candidate_ks:
            kmeans_trial = KMeans(n_clusters=k, random_state=SEED, n_init=20)
            trial_labels = kmeans_trial.fit_predict(embedding_matrix)

            entry = {
                "labels": trial_labels,
                "centers": kmeans_trial.cluster_centers_,
                "inertia": kmeans_trial.inertia_,
                "silhouette": None,
            }

            if len(set(trial_labels)) >= 2:
                entry["silhouette"] = silhouette_score(embedding_matrix, trial_labels)

            results[k] = entry

        valid_sil_ks = [k for k in candidate_ks if results[k]["silhouette"] is not None]

        silhouette_k = None
        if valid_sil_ks:
            silhouette_k = max(valid_sil_ks, key=lambda k: results[k]["silhouette"])

        elbow_k = _elbow_k(candidate_ks, [results[k]["inertia"] for k in candidate_ks])

        if SELECTION_METHOD == "silhouette":
            chosen_k = silhouette_k
        elif SELECTION_METHOD == "elbow":
            chosen_k = elbow_k
        else:  # combined
            if silhouette_k is not None and silhouette_k == elbow_k:
                chosen_k = silhouette_k
            else:
                chosen_k = silhouette_k if silhouette_k is not None else elbow_k

        if chosen_k is None:
            # Every candidate K collapsed to one effective cluster -- fall back.
            n_patterns = 1
            pattern_labels = np.zeros(n_events, dtype=int)
            cluster_centers = embedding_matrix.mean(axis=0, keepdims=True)
        else:
            n_patterns = chosen_k
            pattern_labels = results[chosen_k]["labels"]
            cluster_centers = results[chosen_k]["centers"]

            sil_str = f"{results[chosen_k]['silhouette']:.4f}" if results[chosen_k]["silhouette"] is not None else "n/a"
            print(f"Selection method: {SELECTION_METHOD}")
            print(f"  silhouette-preferred k = {silhouette_k}"
                  + (f" (score {results[silhouette_k]['silhouette']:.4f})" if silhouette_k else ""))
            print(f"  elbow-preferred k      = {elbow_k}")
            print(f"  -> chosen n_patterns   = {n_patterns} (silhouette={sil_str})")

print(f"Latent patterns discovered: {n_patterns}")

# ------------------------------------------------------------
# 4. PATTERN PROBABILITIES
#
# Convert distance from every event to every latent pattern
# into a probability distribution.
# ------------------------------------------------------------

def pattern_probabilities(embedding, centers):
    embedding = embedding / max(np.linalg.norm(embedding), 1e-12)

    centers = centers / np.clip(
        np.linalg.norm(centers, axis=1, keepdims=True),
        1e-12,
        None,
    )

    similarities = centers @ embedding

    # Temperature controls how sharply the event belongs
    # to one pattern.
    temperature = 0.10

    logits = similarities / temperature

    logits = logits - np.max(logits)

    probabilities = np.exp(logits)
    probabilities /= np.sum(probabilities)

    return probabilities

pattern_probability_matrix = np.vstack([
    pattern_probabilities(
        embedding_matrix[i],
        cluster_centers,
    )
    for i in range(n_events)
])

# ------------------------------------------------------------
# 5. TEMPORAL REPRESENTATION
#
# Extract the actual learned temporal message from PHTKG.
#
# This is NOT a hand-written timestamp feature.
# It is produced by the trained temporal parameters:
#
#   linear time component
#   +
#   periodic time component
#
# Unchanged from the model's raw time_message computation --
# this part doesn't depend on entity_state at all, only on the
# year/known columns, so there's nothing to fix here re: chronology.
# ------------------------------------------------------------

def learned_temporal_representation(
    model,
    rows,
    time_mean,
    time_known,
):
    global_ids, years, known, provenance = to_tensors(rows)

    model.eval()

    with torch.inference_mode():

        years_column = years.unsqueeze(-1)
        known_column = known.unsqueeze(-1)

        observed_time = (
            years_column * model.time_linear_weight
            + model.time_linear_bias
            + torch.sin(
                years_column * model.time_periodic_weight
                + model.time_periodic_bias
            )
        )

        time_message = (
            known_column * observed_time
            + (1 - known_column) * model.missing_time
        )

    return time_message.cpu().numpy()

temporal_matrix = learned_temporal_representation(
    predicted_model,
    all_predicted_rows,
    predicted_time_mean,
    predicted_time_known,
)

# ------------------------------------------------------------
# 6. SIMILAR EVENTS
#
# Similarity is cosine similarity in the learned PHTKG
# embedding space.
# ------------------------------------------------------------

similar_events = {}

similarity_matrix = embedding_matrix @ embedding_matrix.T

for i, event_id in enumerate(event_ids):

    ranking = np.argsort(-similarity_matrix[i])

    neighbors = []

    for j in ranking:

        if j == i:
            continue

        neighbors.append({
            "report_id": event_ids[j],
            "similarity": round(
                float(similarity_matrix[i, j]),
                4,
            ),
        })

        if len(neighbors) >= TOP_K_SIMILAR:
            break

    similar_events[event_id] = neighbors

# ------------------------------------------------------------
# 7. NOVELTY SCORE
#
# Novelty is based on how far the event is from the closest
# latent pattern center.
#
# High score = unusual representation.
# Low score  = well represented by an existing pattern.
# ------------------------------------------------------------

novelty_scores = {}

for i, event_id in enumerate(event_ids):

    embedding = embedding_matrix[i]

    similarities = cluster_centers @ embedding

    best_similarity = float(np.max(similarities))

    # Convert similarity into novelty.
    novelty = 1.0 - ((best_similarity + 1.0) / 2.0)

    novelty_scores[event_id] = round(
        float(np.clip(novelty, 0.0, 1.0)),
        4,
    )

# ------------------------------------------------------------
# 7b. RECURRENCE PROBABILITY  (new)
#
# Direct forecast from the trained next_year_head: given the
# entity's state as of now, how likely is it to appear again
# (in any role) next year. Only available on models trained with
# the chronological/recurrence architecture -- skipped (left as
# None) otherwise, rather than faking a number.
# ------------------------------------------------------------

def event_year(event):
    date = event.get("event_date", "")
    if date:
        try:
            return int(str(date)[:4])
        except Exception:
            return None
    return None

recurrence_probabilities = {}

_has_recurrence_head = hasattr(predicted_model, "predict_recurrence") and _entity_state_for_output is not None

if _has_recurrence_head:
    vocab = predicted_graph["vocab"]
    offsets = predicted_graph["offsets"]
    p_year_min = predicted_graph["year_min"]
    p_year_max = predicted_graph["year_max"]

    for event_id in event_ids:
        event = event_lookup.get(event_id) if "event_lookup" in dir() else None
        # event_lookup is built in section 8 below; if this cell is re-run after
        # that point it will exist, otherwise fall back to `predictions`.
        if event is None:
            event = next((e for e in predictions if e["report_id"] == event_id), None)
        if event is None:
            continue

        year = event_year(event) or p_year_max
        target_year = year + 1
        target_norm = (target_year - p_year_min) / max(1, p_year_max - p_year_min)

        role_probabilities = {}
        for role in ["ai_system", "harm_category"]:
            value = normalize(event.get(role, "")) or EMPTY
            local_id = vocab[role].get(value, 0)
            global_id = offsets[role] + local_id
            with torch.inference_mode():
                probability = predicted_model.predict_recurrence(
                    torch.tensor([global_id], device=DEVICE),
                    _entity_state_for_output,
                    target_norm,
                )
            role_probabilities[role] = float(probability[0])

        recurrence_probabilities[event_id] = role_probabilities

    print(f"Computed recurrence probabilities for {len(recurrence_probabilities)} event(s).")
else:
    print("Model has no predict_recurrence (non-chronological checkpoint) -- "
          "recurrence probabilities will be left as null in the output.")

# ------------------------------------------------------------
# 8. CREATE EVENT LOOKUP
# ------------------------------------------------------------

event_lookup = {
    event["report_id"]: event
    for event in predictions
}

# ------------------------------------------------------------
# 9. BUILD FINAL JSON
# ------------------------------------------------------------

final_events = []

for i, event_id in enumerate(event_ids):

    if event_id not in event_lookup:
        continue

    event = event_lookup[event_id]

    probabilities = pattern_probability_matrix[i]

    best_pattern = int(np.argmax(probabilities))

    # Probability assigned to the selected latent pattern.
    best_probability = float(probabilities[best_pattern])

    # --------------------------------------------------------
    # Keep the original event structure.
    # --------------------------------------------------------

    output_event = dict(event)

    # --------------------------------------------------------
    # IMPORTANT:
    # Remove graph_representation if it exists.
    # --------------------------------------------------------

    output_event.pop("graph_representation", None)

    # Single-pass WIP has no contradiction/counterpart stage.
    output_event.pop("counterpart", None)

    # --------------------------------------------------------
    # Learned representation
    # --------------------------------------------------------

    event_recurrence = recurrence_probabilities.get(event_id, {})

    output_event["learned_representation"] = {
        "event_embedding": [
            round(float(x), 6)
            for x in embedding_matrix[i]
        ],

        "pattern_assignment": f"pattern_{best_pattern + 1}",

        "pattern_probability": round(
            best_probability,
            6,
        ),

        "temporal_representation": [
            round(float(x), 6)
            for x in temporal_matrix[i]
        ],

        "similar_events": similar_events[event_id],

        "pattern_similarity": round(
            best_probability,
            6,
        ),

        "novelty_score": novelty_scores[event_id],

        "ai_system_recurrence_probability": event_recurrence.get("ai_system"),
        "harm_category_recurrence_probability": event_recurrence.get("harm_category"),
    }

    final_events.append(output_event)

# ------------------------------------------------------------
# 10. PATTERN EVOLUTION
#
# Patterns are compared across time to determine whether their
# occurrence changes across years.
#
# This is an analysis of the learned latent patterns.
#
# frequency_trend added: linear slope of event-count-per-year
# within the pattern. This is the concrete "is it recurring too
# much" signal -- increasing / decreasing / stable / insufficient_data.
# ------------------------------------------------------------

def frequency_trend_for(years_list):
    counts_by_year = {}
    for year in years_list:
        counts_by_year[year] = counts_by_year.get(year, 0) + 1

    distinct_years = sorted(counts_by_year)
    if len(distinct_years) < 2:
        return {
            "counts_by_year": counts_by_year,
            "slope_events_per_year": None,
            "trend": "insufficient_data",
        }

    xs = np.array(distinct_years, dtype=float)
    ys = np.array([counts_by_year[year] for year in distinct_years], dtype=float)
    slope = float(np.polyfit(xs, ys, 1)[0])

    if slope > FREQUENCY_TREND_THRESHOLD:
        trend = "increasing"
    elif slope < -FREQUENCY_TREND_THRESHOLD:
        trend = "decreasing"
    else:
        trend = "stable"

    return {
        "counts_by_year": counts_by_year,
        "slope_events_per_year": slope,
        "trend": trend,
    }

pattern_evolution = {}

for pattern_index in range(n_patterns):

    members = []

    for i, event_id in enumerate(event_ids):

        if pattern_labels[i] == pattern_index:

            event = event_lookup.get(event_id)

            if event is None:
                continue

            date = event.get("event_date", "")

            year = None

            if date:
                try:
                    year = int(str(date)[:4])
                except Exception:
                    year = None

            members.append({
                "report_id": event_id,
                "year": year,
            })

    years = [
        m["year"]
        for m in members
        if m["year"] is not None
    ]

    pattern_evolution[f"pattern_{pattern_index + 1}"] = {
        "event_count": len(members),
        "years": sorted(years),
        "events": members,
        "frequency_trend": frequency_trend_for(years),
    }

# ------------------------------------------------------------
# 11. ATTACH PATTERN EVOLUTION INFORMATION
# ------------------------------------------------------------

for event in final_events:

    pattern_name = event[
        "learned_representation"
    ]["pattern_assignment"]

    event["learned_representation"][
        "pattern_evolution"
    ] = pattern_evolution.get(
        pattern_name,
        {
            "event_count": 0,
            "years": [],
            "events": [],
            "frequency_trend": {
                "counts_by_year": {},
                "slope_events_per_year": None,
                "trend": "insufficient_data",
            },
        },
    )

# ------------------------------------------------------------
# 12. SAVE FINAL JSONL
# ------------------------------------------------------------

with FINAL_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    for event in final_events:

        f.write(
            json.dumps(
                event,
                ensure_ascii=False,
            )
            + "\n"
        )

# ------------------------------------------------------------
# 13. SUMMARY
# ------------------------------------------------------------

print()
print("=" * 70)
print("FINAL PHTKG LEARNED REPRESENTATION CREATED")
print("=" * 70)

print(f"Events exported: {len(final_events)}")
print(f"Embedding dimension: {embedding_matrix.shape[1]}")
print(f"Latent patterns: {n_patterns}")
print(f"Similar events per event: {TOP_K_SIMILAR}")
print(f"Recurrence probabilities computed: {_has_recurrence_head}")
print(f"Output: {FINAL_OUTPUT_PATH}")

In [ ]:
# Convert final JSONL to a formatted JSON array for inspection/export.
json_array = []
if FINAL_OUTPUT_PATH.exists():
    with FINAL_OUTPUT_PATH.open("r", encoding="utf-8") as infile:
        json_array = [json.loads(line) for line in infile if line.strip()]

FINAL_JSON_PATH = FINAL_OUTPUT_PATH.with_suffix(".json")
with FINAL_JSON_PATH.open("w", encoding="utf-8") as outfile:
    json.dump(json_array, outfile, ensure_ascii=False, indent=2)

print("Saved:", FINAL_JSON_PATH)
